In [0]:
# ═══════════════════════════════════════════════════════════════
# NOTEBOOK 01: ENVIRONMENT & DATA INGESTION
# Membership Churn Analysis - Dissertation Project
# ═══════════════════════════════════════════════════════════════

print("🚀 Notebook 01: Environment & Data Ingestion")
print("=" * 70)
print("Author: D. Dublin")
print("Date: February 15, 2026")
print("Dataset: churn_t_db.csv (2.1GB, 18.4M rows)")
print("=" * 70)

🚀 Notebook 01: Environment & Data Ingestion
Author: D. Dublin
Date: February 15, 2026
Dataset: churn_t_db.csv (2.1GB, 18.4M rows)


# Membership Churn Analysis

## Notebook 01: Environment & Data Ingestion

---

**Author:** D. 
**Date:** February 15, 2026 
**Dataset:** churn_t_db.csv (2.1GB, 18.4M rows)

---

## 1.0 Data Ingestion

### 1.1 Environment Verification

**CONTEXT**

To establish the foundational environment for analyzing the Organisation's membership churn dataset. This includes verifying Spark availability, confirming file access, and validating the uploaded CSV before any transformations begin.

**PURPOSE**

To ensure:
1. PySpark environment is properly initialized
2. The uploaded CSV file is accessible at the expected location
3. File size matches expectations (~2.1GB)
4. Confident progression to data ingestion

**STEP**

Confirm that `churn_t_db.csv` exists at the expected Unity Catalog volume path and compute its size in gigabytes. Verify PySpark environment initialization.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Environment & File Verification
# ═══════════════════════════════════════════════════════════════

# Check Spark version
print(f"Spark Version: {spark.version}")
print("-" * 70)

# Define file path
file_path = "/Volumes/workspace/rcn_churn/raw_data/churn_t_db.csv"

# Verify file exists and get details
file_info = dbutils.fs.ls("/Volumes/workspace/rcn_churn/raw_data/")

# Display file information
display(file_info)

# Calculate and print file size
for file in file_info:
    if "churn_t_db" in file.name:
        size_gb = file.size / (1024**3)
        print(f"\nFile: {file.name}")
        print(f"Size: {size_gb:.2f} GB")

Spark Version: 4.1.0
----------------------------------------------------------------------


path,name,size,modificationTime
dbfs:/Volumes/workspace/rcn_churn/raw_data/churn_t_db.csv,churn_t_db.csv,2216363321,1771178702000
dbfs:/Volumes/workspace/rcn_churn/raw_data/delta_raw/,delta_raw/,0,1771356901534



File: churn_t_db.csv
Size: 2.06 GB


**RESULT**

The file was located successfully at `/Volumes/workspace/rcn_churn/raw_data/churn_t_db.csv` and measures 2.06 GB in size. This confirms correct path configuration and establishes the scale of the dataset prior to ingestion. PySpark environment is initialized and operational.

**Status:** ✓ Pass

### 1.2 Data Loading

**CONTEXT**

The raw dataset is loaded without transformation to preserve its original structure at the point of ingestion. Given the file size (2.06 GB), PySpark's distributed processing capabilities are utilized for memory-efficient reading.

**PURPOSE**

To ingest the dataset in a controlled manner and confirm its dimensionality prior to structural validation.

**STEP**

Read `churn_t_db.csv` using PySpark and return row and column counts.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Load CSV with PySpark
# ═══════════════════════════════════════════════════════════════

# Read CSV with schema inference
df_raw = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

# Get dimensions
row_count = df_raw.count()
col_count = len(df_raw.columns)

# Display results
print(f"Rows: {row_count:,}")
print(f"Columns: {col_count}")
print("-" * 70)

# Display first few rows
display(df_raw.limit(5))

Rows: 18,461,480
Columns: 12
----------------------------------------------------------------------


_c0,CM_snapshot_date,Int_nurse,Region,MemCategory,CatName,Branch,YoB,MemSectorType,YoJ,q_members_t,q_leavers_t
0,2021-01-01,EEU,East Midlands,Nurse member,Joint - RCN/RCM,Nottingham,1993.0,NHS,2017.0,2.969121140142518,null
1,2021-01-01,EEU,East Midlands,Nurse member,Nurse - 1st Year Discount,Derbyshire,1990.0,Independent,2020.0,2.969121140142518,null
2,2021-01-01,EEU,East Midlands,Nurse member,Nurse - 1st Year Discount,Leicestershire & Rutland,1989.0,NHS,2020.0,2.969121140142518,null
3,2021-01-01,EEU,East Midlands,Nurse member,Nurse - 1st Year Discount,Northamptonshire,1990.0,null,2020.0,2.969121140142518,null
4,2021-01-01,EEU,East Midlands,Nurse member,Nurse - 1st Year Discount,Northamptonshire,1990.0,Independent,2020.0,2.969121140142518,null


**RESULT**

The dataset was successfully loaded with 18,461,480 rows and 12 columns. PySpark inferred schema types automatically, though several columns require type correction: `YoB` and `YoJ` were interpreted as `double` but represent integer year values. Initial inspection reveals `q_leavers_t` contains null values in the first five rows which requires further investigation, and `MemSectorType` shows missingness. Column names and structure are present.

**Status:** ✓ Pass

### 1.3 Schema Validation

**CONTEXT**

Schema validation confirms column names, data types, and structure of the ingested dataset. This step identifies the inferred types and any potential mismatches before proceeding to transformation.

**PURPOSE**

To verify:
1. Column names and count
2. Inferred data types for each column
3. Any type corrections required before data storage and transformation

**STEP**

Display the complete schema with column names and inferred data types.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Schema Inspection
# ═══════════════════════════════════════════════════════════════

# Print schema in tree format
print("Inferred Schema:")
print("-" * 70)
df_raw.printSchema()

# List all column names
print("\nColumn Names:")
print("-" * 70)
for i, col in enumerate(df_raw.columns, 1):
    print(f"{i}. {col}")

Inferred Schema:
----------------------------------------------------------------------
root
 |-- _c0: integer (nullable = true)
 |-- CM_snapshot_date: date (nullable = true)
 |-- Int_nurse: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- MemCategory: string (nullable = true)
 |-- CatName: string (nullable = true)
 |-- Branch: string (nullable = true)
 |-- YoB: double (nullable = true)
 |-- MemSectorType: string (nullable = true)
 |-- YoJ: double (nullable = true)
 |-- q_members_t: double (nullable = true)
 |-- q_leavers_t: double (nullable = true)


Column Names:
----------------------------------------------------------------------
1. _c0
2. CM_snapshot_date
3. Int_nurse
4. Region
5. MemCategory
6. CatName
7. Branch
8. YoB
9. MemSectorType
10. YoJ
11. q_members_t
12. q_leavers_t


**RESULT**

The dataset contains 12 columns with naming conventions intact. Schema inference identified two type assignments: `YoB` and `YoJ` are typed as `double` but appear to represent year values based on sample data. All columns are marked as nullable by the inference process.

**Status:** ✓ Pass

### 1.4 Raw Data Table Creation

**CONTEXT**

The ingested CSV data is persisted in Delta table format to enable efficient access for subsequent analysis. Delta tables provide faster read performance and support data versioning compared to repeatedly reading the original CSV file.

**PURPOSE**

To save the raw ingested data in Delta format for efficient downstream processing and establish a documented baseline before any transformations.

**STEP**

Write the raw DataFrame to Delta format at a designated storage location and verify successful table creation.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Save Raw Data as Delta Table
# ═══════════════════════════════════════════════════════════════

# Define Delta table path
delta_raw_path = "/Volumes/workspace/rcn_churn/raw_data/delta_raw"

# Write to Delta format
df_raw.write.format("delta").mode("overwrite").save(delta_raw_path)

print(f"✓ Raw data table created at: {delta_raw_path}")
print("-" * 70)

# Verify table creation
display(dbutils.fs.ls("/Volumes/workspace/rcn_churn/raw_data/"))

✓ Raw data table created at: /Volumes/workspace/rcn_churn/raw_data/delta_raw
----------------------------------------------------------------------


path,name,size,modificationTime
dbfs:/Volumes/workspace/rcn_churn/raw_data/churn_t_db.csv,churn_t_db.csv,2216363321,1771178702000
dbfs:/Volumes/workspace/rcn_churn/raw_data/delta_raw/,delta_raw/,0,1771356958138


**RESULT**

Raw data table successfully created in Delta format at `/Volumes/workspace/rcn_churn/raw_data/delta_raw/`. Verification confirms the Delta directory exists alongside the original CSV file (2.06 GB). The raw data is now persisted and ready for quality assessment.

**Status:** ✓ Pass

## 2.0 Data Quality Assessment

### 2.1 Completeness Analysis

**CONTEXT**

Completeness analysis quantifies missing data across all columns to identify patterns of missingness and assess data availability for subsequent analysis. Understanding which variables contain null values and their prevalence informs data handling decisions.

**PURPOSE**

To determine:
1. Which columns contain missing values
2. The count and percentage of missing values per column
3. Overall data completeness across the dataset

**STEP**

Calculate the count and percentage of null values for each column in the dataset and display the results in a structured format.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Missingness Analysis
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import col, count, when, round as spark_round

# Calculate total rows
total_rows = df_raw.count()

# Calculate null counts for each column
null_counts = df_raw.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_raw.columns
])

# Convert to list for easier processing
null_list = null_counts.collect()[0].asDict()

# Create results list
results = []
for column_name in df_raw.columns:
    null_count = null_list[column_name]
    null_pct = (null_count / total_rows) * 100
    results.append({
        'Column': column_name,
        'Null_Count': null_count,
        'Null_Percentage': round(null_pct, 4)
    })

# Convert to DataFrame for display
df_missingness = spark.createDataFrame(results)
display(df_missingness)

Column,Null_Count,Null_Percentage
_c0,0,0.0
CM_snapshot_date,0,0.0
Int_nurse,0,0.0
Region,1,0.0
MemCategory,0,0.0
CatName,0,0.0
Branch,0,0.0
YoB,4406,0.0239
MemSectorType,1436695,7.7821
YoJ,39,2.0E-4


**RESULT**

Missingness analysis reveals significant variation in data completeness across columns. Nine columns show complete data (0% missing). Three columns show minimal missingness: `Region` (1 record), `YoJ` (39 records, 0.0002%), and `YoB` (4,406 records, 0.0239%). Two columns show substantial missingness: `MemSectorType` (1,436,695 records, 7.78%) and `q_leavers_t` (18,259,005 records, 98.90%). The near-complete absence of `q_leavers_t` values warrants investigation into whether this represents true missingness or a data collection issue.

**Status:** ⚠️ Investigate

### 2.2 Cardinality Profiling

**PURPOSE**

To determine:
1. The number of distinct values in each column
2. The ratio of distinct values to total records
3. Which columns are categorical versus continuous in nature

**STEP**

Calculate the count of distinct values for each column and compute the distinctness ratio (distinct values / total rows) to assess cardinality characteristics.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Cardinality Analysis
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import countDistinct

# Calculate distinct counts for each column
distinct_counts = df_raw.select([
    countDistinct(col(c)).alias(c) 
    for c in df_raw.columns
])

# Convert to list
distinct_list = distinct_counts.collect()[0].asDict()

# Create results list
cardinality_results = []
for column_name in df_raw.columns:
    distinct_count = distinct_list[column_name]
    distinctness_ratio = (distinct_count / total_rows) * 100
    cardinality_results.append({
        'Column': column_name,
        'Distinct_Count': distinct_count,
        'Distinctness_Ratio_%': round(distinctness_ratio, 4)
    })

# Convert to DataFrame for display
df_cardinality = spark.createDataFrame(cardinality_results)
display(df_cardinality)

Column,Distinct_Count,Distinctness_Ratio_%
_c0,18461480,100.0
CM_snapshot_date,60,3.0E-4
Int_nurse,4,0.0
Region,14,1.0E-4
MemCategory,4,0.0
CatName,17,1.0E-4
Branch,104,6.0E-4
YoB,105,6.0E-4
MemSectorType,5,0.0
YoJ,85,5.0E-4


**RESULT**

Cardinality analysis reveals clear categorical structure in the dataset. The `_c0` column shows 100% distinctness (18,461,480 unique values), confirming it functions as a unique row identifier. Low-cardinality categorical variables include: `Int_nurse` (4 distinct), `MemCategory` (4 distinct), `MemSectorType` (5 distinct), and `Region` (14 distinct). Temporal and cohort variables show moderate cardinality: `CM_snapshot_date` (60 distinct), `YoB` (105 distinct), and `YoJ` (85 distinct). The `Branch` variable contains 104 distinct values. Notably, `q_members_t` shows 99 distinct values and `q_leavers_t` shows only 11 distinct values despite being numeric measures, suggesting these may be aggregated or binned values rather than raw counts.

**Status:** ✓ Pass

### 2.3 Data Type Consistency Validation

**CONTEXT**

Data type consistency validation examines whether values within each column conform to their inferred types and expected patterns. This systematic inspection across all 18.4 million rows identifies type inconsistencies, unexpected patterns, or values outside reasonable bounds that may indicate data quality issues.

**PURPOSE**

To verify:
1. YoB and YoJ contain only whole number values despite being typed as double
2. Value ranges for year columns fall within plausible bounds
3. Date values conform to expected temporal range
4. Numeric columns contain valid values without unexpected patterns

**STEP**

Calculate minimum and maximum values for numeric and date columns, check for decimal values in year columns, and assess whether observed ranges align with expected data characteristics.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Data Type Consistency Validation
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import min, max, col

print("VALUE RANGE ANALYSIS")
print("=" * 70)

# Analyze numeric and date columns
columns_to_check = ['_c0', 'CM_snapshot_date', 'YoB', 'YoJ', 'q_members_t', 'q_leavers_t']

for col_name in columns_to_check:
    print(f"\n{col_name}:")
    print("-" * 70)
    
    # Get min and max
    result = df_raw.select(
        min(col(col_name)).alias('min_value'),
        max(col(col_name)).alias('max_value')
    ).collect()[0]
    
    print(f"Min: {result['min_value']}")
    print(f"Max: {result['max_value']}")

print("\n" + "=" * 70)
print("DECIMAL CHECK FOR YEAR COLUMNS")
print("=" * 70)

# Check if YoB and YoJ have decimal parts
for year_col in ['YoB', 'YoJ']:
    print(f"\n{year_col}:")
    print("-" * 70)
    
    # Check if any values have decimal parts (value != floor(value))
    decimal_check = df_raw.filter(
        (col(year_col).isNotNull()) & 
        (col(year_col) != col(year_col).cast('integer'))
    ).count()
    
    print(f"Rows with decimal values: {decimal_check}")

VALUE RANGE ANALYSIS

_c0:
----------------------------------------------------------------------
Min: 0
Max: 18461479

CM_snapshot_date:
----------------------------------------------------------------------
Min: 2021-01-01
Max: 2025-12-01

YoB:
----------------------------------------------------------------------
Min: 1895.0
Max: 2966.0

YoJ:
----------------------------------------------------------------------
Min: 1941.0
Max: 2025.0

q_members_t:
----------------------------------------------------------------------
Min: 2.969121140142518
Max: 296.9121140142518

q_leavers_t:
----------------------------------------------------------------------
Min: 2.969121140142518
Max: 50.475059382422806

DECIMAL CHECK FOR YEAR COLUMNS

YoB:
----------------------------------------------------------------------
Rows with decimal values: 0

YoJ:
----------------------------------------------------------------------
Rows with decimal values: 0


**RESULT**

Value range analysis confirms type consistency across numeric and date columns. The `_c0` column ranges from 0 to 18,461,479, functioning as a sequential identifier. Date values span from 2021-01-01 to 2025-12-01. Year-format columns (`YoB` and `YoJ`) contain no decimal values (0 rows with fractional parts), confirming they can be safely cast to integer. `YoB` ranges from 1895 to 2966 with 105 distinct values. `YoJ` ranges from 1941 to 2025 with 85 distinct values. The quantitative measures show: `q_members_t` ranges from 2.97 to 296.91 with 99 distinct values, and `q_leavers_t` ranges from 2.97 to 50.48 with 11 distinct values.

**Patterns Observed:**

The date column contains 60 distinct values across a five-year span (2021-2025), suggesting monthly snapshot intervals (5 years × 12 months = 60). The `YoJ` range (1941-2025) spans exactly 84 years, and with 85 distinct values (including both endpoints), this suggests continuous annual representation. In contrast, `YoB` spans 1,071 years but contains only 105 distinct values, indicating sparse representation with a notable outlier at 2966. Both quantitative measures share the same minimum value (2.97): `q_members_t` shows an approximately 100× range multiplier (296.91 / 2.97 ≈ 100) with 99 distinct values, while `q_leavers_t` shows an approximately 17× range multiplier (50.48 / 2.97 ≈ 17) with only 11 distinct values. This mathematical relationship in both columns suggests these may be calculated or weighted values rather than raw counts.

**Status:** ⚠️ Investigate

### 2.4 Numerical Value Enumeration

**CONTEXT**

Numerical value enumeration lists all distinct values for numerical columns with low cardinality to reveal patterns, validate the mathematical relationships observed in range analysis, and identify any anomalous values requiring investigation.

**PURPOSE**

To enumerate:
1. All distinct values for numerical columns with fewer than 110 distinct entries
2. Patterns in the actual values (sequences, gaps, calculated relationships)
3. Confirmation of observed mathematical patterns from range analysis

**STEP**

Extract and display all distinct values for `CM_snapshot_date`, `YoB`, `YoJ`, `q_members_t`, and `q_leavers_t` in sorted order to examine the complete value distributions.

#### 2.4.1 Distinct Values - CM_snapshot_date

In [0]:
# ═══════════════════════════════════════════════════════════════
# CM_snapshot_date - Distinct Values
# ═══════════════════════════════════════════════════════════════

print("CM_snapshot_date - All Distinct Values:")
print("-" * 70)

distinct_dates = df_raw.select('CM_snapshot_date').distinct().orderBy('CM_snapshot_date')
display(distinct_dates)

CM_snapshot_date - All Distinct Values:
----------------------------------------------------------------------


CM_snapshot_date
2021-01-01
2021-02-01
2021-03-01
2021-04-01
2021-05-01
2021-06-01
2021-07-01
2021-08-01
2021-09-01
2021-10-01


**Sequence Validation - CM_snapshot_date:**

Programmatically verify that the snapshot dates form a complete monthly sequence with no gaps.

In [0]:
# ═══════════════════════════════════════════════════════════════
# CM_snapshot_date - Sequence Validation
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import year, month

print("CM_snapshot_date VALIDATION")
print("=" * 70)

# Get min and max dates
date_range = df_raw.select(
    min('CM_snapshot_date').alias('min_date'),
    max('CM_snapshot_date').alias('max_date')
).collect()[0]

min_date = date_range['min_date']
max_date = date_range['max_date']

print(f"\nDate Range: {min_date} to {max_date}")

# Calculate expected number of months
months_diff = (max_date.year - min_date.year) * 12 + (max_date.month - min_date.month) + 1
print(f"Expected months (inclusive): {months_diff}")

# Get actual distinct count
actual_count = df_raw.select('CM_snapshot_date').distinct().count()
print(f"Actual distinct dates: {actual_count}")

# Check if all dates are the 1st of the month
non_first_dates = df_raw.filter("day(CM_snapshot_date) != 1").count()
print(f"Dates not on 1st of month: {non_first_dates}")

# Validate completeness
if actual_count == months_diff and non_first_dates == 0:
    print("\n✓ VALIDATION PASSED: Complete monthly sequence with all dates on the 1st")
else:
    print("\n⚠️ VALIDATION ISSUE: Missing months or non-standard dates detected")

CM_snapshot_date VALIDATION

Date Range: 2021-01-01 to 2025-12-01
Expected months (inclusive): 60
Actual distinct dates: 60
Dates not on 1st of month: 0

✓ VALIDATION PASSED: Complete monthly sequence with all dates on the 1st


#### 2.4.2 Distinct Values - YoB

In [0]:
# ═══════════════════════════════════════════════════════════════
# YoB - Distinct Values
# ═══════════════════════════════════════════════════════════════

print("YoB - All Distinct Values:")
print("-" * 70)

distinct_yob = df_raw.select('YoB').distinct().orderBy('YoB')
display(distinct_yob)

YoB - All Distinct Values:
----------------------------------------------------------------------


YoB
null
1895.0
1900.0
1908.0
1914.0
1915.0
1917.0
1918.0
1919.0
1920.0


**Pattern Analysis - YoB:**

Examine the distribution of birth years, identify gaps in the sequence, and flag implausible values.

In [0]:
# ═══════════════════════════════════════════════════════════════
# YoB - Pattern Analysis
# ═══════════════════════════════════════════════════════════════

print("YoB PATTERN ANALYSIS")
print("=" * 70)

# Get all distinct YoB values (excluding null)
yob_values = df_raw.select('YoB').distinct().filter('YoB is not null').orderBy('YoB').collect()
yob_list = [int(row['YoB']) for row in yob_values]

print(f"\nTotal distinct YoB values (excluding null): {len(yob_list)}")

# Use Python's built-in min/max by specifying them explicitly
import builtins
print(f"Range: {builtins.min(yob_list)} to {builtins.max(yob_list)}")

# Identify implausible future years (> 2026, current year + 1)
future_years = [y for y in yob_list if y > 2026]
print(f"\nImplausible future years (> 2026): {len(future_years)}")
print(f"Values: {future_years}")

# Identify very recent years (2020-2026) - would make members 0-6 years old
recent_years = [y for y in yob_list if 2020 <= y <= 2026]
print(f"\nRecent years (2020-2026) - members aged 0-6: {len(recent_years)}")
print(f"Values: {recent_years}")

# Check sequential coverage from 1917-2008
sequential_range = list(range(1917, 2009))
missing_years = [y for y in sequential_range if y not in yob_list]
print(f"\nMissing years in 1917-2008 range: {len(missing_years)}")
if missing_years:
    print(f"Missing: {missing_years}")
else:
    print("✓ Complete sequential coverage 1917-2008")

# Sparse early years
early_years = [y for y in yob_list if y < 1917]
print(f"\nSparse early years (< 1917): {len(early_years)}")
print(f"Values: {early_years}")

YoB PATTERN ANALYSIS

Total distinct YoB values (excluding null): 105
Range: 1895 to 2966

Implausible future years (> 2026): 4
Values: [2029, 2042, 2044, 2966]

Recent years (2020-2026) - members aged 0-6: 4
Values: [2020, 2021, 2023, 2024]

Missing years in 1917-2008 range: 0
✓ Complete sequential coverage 1917-2008

Sparse early years (< 1917): 5
Values: [1895, 1900, 1908, 1914, 1915]


#### 2.4.3 Distinct Values - YoJ

In [0]:
# ═══════════════════════════════════════════════════════════════
# YoJ - Distinct Values
# ═══════════════════════════════════════════════════════════════

print("YoJ - All Distinct Values:")
print("-" * 70)

distinct_yoj = df_raw.select('YoJ').distinct().orderBy('YoJ')
display(distinct_yoj)

YoJ - All Distinct Values:
----------------------------------------------------------------------


YoJ
null
1941.0
1942.0
1943.0
1944.0
1945.0
1946.0
1947.0
1948.0
1949.0


**Pattern Analysis - YoJ:**

Verify sequential completeness from 1941-2025 and quantify null values.

In [0]:
# ═══════════════════════════════════════════════════════════════
# YoJ - Pattern Analysis
# ═══════════════════════════════════════════════════════════════

import builtins

print("YoJ PATTERN ANALYSIS")
print("=" * 70)

# Get all distinct YoJ values (excluding null)
yoj_values = df_raw.select('YoJ').distinct().filter('YoJ is not null').orderBy('YoJ').collect()
yoj_list = [int(row['YoJ']) for row in yoj_values]

print(f"\nTotal distinct YoJ values (excluding null): {len(yoj_list)}")
print(f"Range: {builtins.min(yoj_list)} to {builtins.max(yoj_list)}")

# Check if sequential
expected_range = list(range(1941, 2026))  # 1941 to 2025 inclusive
missing_years = [y for y in expected_range if y not in yoj_list]

print(f"\nExpected years (1941-2025): {len(expected_range)}")
print(f"Missing years: {len(missing_years)}")

if missing_years:
    print(f"Missing: {missing_years}")
else:
    print("✓ Complete sequential coverage 1941-2025")

# Count null values
null_count = df_raw.filter('YoJ is null').count()
print(f"\nNull values: {null_count}")
print(f"Null percentage: {(null_count / total_rows) * 100:.4f}%")

YoJ PATTERN ANALYSIS

Total distinct YoJ values (excluding null): 85
Range: 1941 to 2025

Expected years (1941-2025): 85
Missing years: 0
✓ Complete sequential coverage 1941-2025

Null values: 39
Null percentage: 0.0002%


#### 2.4.4 Distinct Values - q_members_t

In [0]:
# ═══════════════════════════════════════════════════════════════
# q_members_t - Distinct Values
# ═══════════════════════════════════════════════════════════════

print("q_members_t - All Distinct Values:")
print("-" * 70)

distinct_qmembers = df_raw.select('q_members_t').distinct().orderBy('q_members_t')
display(distinct_qmembers)

q_members_t - All Distinct Values:
----------------------------------------------------------------------


q_members_t
2.969121140142518
5.938242280285036
8.907363420427554
11.876484560570072
14.84560570071259
17.81472684085511
20.783847980997624
23.752969121140143
26.722090261282663
29.69121140142518


**Pattern Analysis - q_members_t:**

Examine the mathematical relationship between distinct values and test the hypothesis of a base multiplier pattern.

In [0]:
# ═══════════════════════════════════════════════════════════════
# q_members_t - Pattern Analysis
# ═══════════════════════════════════════════════════════════════

import builtins

print("q_members_t PATTERN ANALYSIS")
print("=" * 70)

# Get all distinct values
qmembers_values = df_raw.select('q_members_t').distinct().orderBy('q_members_t').collect()
qmembers_list = [row['q_members_t'] for row in qmembers_values]

print(f"\nTotal distinct values: {len(qmembers_list)}")
print(f"Range: {builtins.min(qmembers_list):.2f} to {builtins.max(qmembers_list):.2f}")

# Test base value hypothesis
base_value = qmembers_list[0]
print(f"\nBase value (minimum): {base_value}")

# Calculate ratios to base value
ratios = [val / base_value for val in qmembers_list]
rounded_ratios = [round(r) for r in ratios]

print(f"\nRatio range: {builtins.min(ratios):.2f} to {builtins.max(ratios):.2f}")
print(f"Rounded ratios range: {builtins.min(rounded_ratios)} to {builtins.max(rounded_ratios)}")

# Check if rounded ratios are sequential
expected_multipliers = list(range(1, 101))  # 1 to 100
actual_unique_rounded = sorted(set(rounded_ratios))

print(f"\nExpected multipliers (1-100): {len(expected_multipliers)}")
print(f"Actual unique rounded multipliers: {len(actual_unique_rounded)}")

# Check for gaps
missing_multipliers = [m for m in expected_multipliers if m not in actual_unique_rounded]
if missing_multipliers:
    print(f"Missing multipliers: {missing_multipliers}")
else:
    print("✓ Complete sequential multipliers 1-100")

# Show first 10 and last 10 values with their multipliers
print("\nFirst 10 values:")
for i in range(10):
    print(f"  {qmembers_list[i]:.6f} = {base_value:.6f} × {ratios[i]:.2f} (≈{rounded_ratios[i]})")

print("\nLast 10 values:")
for i in range(-10, 0):
    print(f"  {qmembers_list[i]:.6f} = {base_value:.6f} × {ratios[i]:.2f} (≈{rounded_ratios[i]})")

q_members_t PATTERN ANALYSIS

Total distinct values: 99
Range: 2.97 to 296.91

Base value (minimum): 2.969121140142518

Ratio range: 1.00 to 100.00
Rounded ratios range: 1 to 100

Expected multipliers (1-100): 100
Actual unique rounded multipliers: 99
Missing multipliers: [99]

First 10 values:
  2.969121 = 2.969121 × 1.00 (≈1)
  5.938242 = 2.969121 × 2.00 (≈2)
  8.907363 = 2.969121 × 3.00 (≈3)
  11.876485 = 2.969121 × 4.00 (≈4)
  14.845606 = 2.969121 × 5.00 (≈5)
  17.814727 = 2.969121 × 6.00 (≈6)
  20.783848 = 2.969121 × 7.00 (≈7)
  23.752969 = 2.969121 × 8.00 (≈8)
  26.722090 = 2.969121 × 9.00 (≈9)
  29.691211 = 2.969121 × 10.00 (≈10)

Last 10 values:
  267.220903 = 2.969121 × 90.00 (≈90)
  270.190024 = 2.969121 × 91.00 (≈91)
  273.159145 = 2.969121 × 92.00 (≈92)
  276.128266 = 2.969121 × 93.00 (≈93)
  279.097387 = 2.969121 × 94.00 (≈94)
  282.066508 = 2.969121 × 95.00 (≈95)
  285.035629 = 2.969121 × 96.00 (≈96)
  288.004751 = 2.969121 × 97.00 (≈97)
  290.973872 = 2.969121 × 98.00 (≈

#### 2.4.5 Distinct Values - q_leavers_t

In [0]:
# ═══════════════════════════════════════════════════════════════
# q_leavers_t - Distinct Values
# ═══════════════════════════════════════════════════════════════

print("q_leavers_t - All Distinct Values:")
print("-" * 70)

distinct_qleavers = df_raw.select('q_leavers_t').distinct().orderBy('q_leavers_t')
display(distinct_qleavers)

q_leavers_t - All Distinct Values:
----------------------------------------------------------------------


q_leavers_t
null
2.969121140142518
5.938242280285036
8.907363420427554
11.876484560570072
14.84560570071259
17.81472684085511
20.783847980997624
23.752969121140143
26.722090261282663


**Pattern Analysis - q_leavers_t:**

Examine the mathematical relationship, identify the gap in multipliers, and quantify null values.

In [0]:
# ═══════════════════════════════════════════════════════════════
# q_leavers_t - Pattern Analysis
# ═══════════════════════════════════════════════════════════════

import builtins

print("q_leavers_t PATTERN ANALYSIS")
print("=" * 70)

# Get all distinct non-null values
qleavers_values = df_raw.select('q_leavers_t').distinct().filter('q_leavers_t is not null').orderBy('q_leavers_t').collect()
qleavers_list = [row['q_leavers_t'] for row in qleavers_values]

print(f"\nTotal distinct values (excluding null): {len(qleavers_list)}")
print(f"Range: {builtins.min(qleavers_list):.2f} to {builtins.max(qleavers_list):.2f}")

# Test if using same base value as q_members_t
base_value = 2.969121140142518
print(f"\nBase value (from q_members_t): {base_value}")

# Calculate multipliers
multipliers = [val / base_value for val in qleavers_list]
rounded_multipliers = [round(m) for m in multipliers]

print(f"\nMultiplier range: {builtins.min(multipliers):.2f} to {builtins.max(multipliers):.2f}")
print(f"\nActual multipliers present:")
for i, val in enumerate(qleavers_list):
    print(f"  {val:.6f} = {base_value:.6f} × {multipliers[i]:.2f} (≈{rounded_multipliers[i]})")

# Identify the gap
expected_continuous = list(range(1, builtins.max(rounded_multipliers) + 1))
missing_multipliers = [m for m in expected_continuous if m not in rounded_multipliers]

print(f"\nExpected continuous range (1-17): {len(expected_continuous)}")
print(f"Actual multipliers present: {len(rounded_multipliers)}")
print(f"Missing multipliers: {missing_multipliers}")

# Count null values
null_count = df_raw.filter('q_leavers_t is null').count()
print(f"\nNull values: {null_count:,}")
print(f"Null percentage: {(null_count / total_rows) * 100:.2f}%")

# Distribution of non-null values
print("\n" + "=" * 70)
print("VALUE DISTRIBUTION (non-null records only)")
print("=" * 70)

value_counts = df_raw.filter('q_leavers_t is not null').groupBy('q_leavers_t').count().orderBy('q_leavers_t')
display(value_counts)

print(f"\nTotal non-null records: {df_raw.filter('q_leavers_t is not null').count():,}")

q_leavers_t PATTERN ANALYSIS

Total distinct values (excluding null): 11
Range: 2.97 to 50.48

Base value (from q_members_t): 2.969121140142518

Multiplier range: 1.00 to 17.00

Actual multipliers present:
  2.969121 = 2.969121 × 1.00 (≈1)
  5.938242 = 2.969121 × 2.00 (≈2)
  8.907363 = 2.969121 × 3.00 (≈3)
  11.876485 = 2.969121 × 4.00 (≈4)
  14.845606 = 2.969121 × 5.00 (≈5)
  17.814727 = 2.969121 × 6.00 (≈6)
  20.783848 = 2.969121 × 7.00 (≈7)
  23.752969 = 2.969121 × 8.00 (≈8)
  26.722090 = 2.969121 × 9.00 (≈9)
  29.691211 = 2.969121 × 10.00 (≈10)
  50.475059 = 2.969121 × 17.00 (≈17)

Expected continuous range (1-17): 17
Actual multipliers present: 11
Missing multipliers: [11, 12, 13, 14, 15, 16]

Null values: 18,259,005
Null percentage: 98.90%

VALUE DISTRIBUTION (non-null records only)


q_leavers_t,count
2.969121140142518,197390
5.938242280285036,4357
8.907363420427554,501
11.876484560570072,139
14.84560570071259,49
17.81472684085511,16
20.783847980997624,9
23.752969121140143,7
26.722090261282663,5
29.69121140142518,1



Total non-null records: 202,475


**RESULT**

Numerical value enumeration reveals distinct patterns across columns:

**2.4.1 CM_snapshot_date:**
- 60 distinct monthly snapshots from 2021-01-01 to 2025-12-01
- Complete sequential monthly coverage confirmed (no missing months)
- All dates fall on the 1st of the month
- Validates monthly aggregation structure

**2.4.2 YoB (Year of Birth):**
- 105 distinct values ranging from 1895 to 2966
- Complete sequential coverage 1917-2008 (92 consecutive years)
- 5 sparse early years: 1895, 1900, 1908, 1914, 1915
- 4 implausible future years: 2029, 2042, 2044, 2966 (data quality issues)
- 4 recent years (2020-2026) suggesting members aged 0-6 years (requires investigation)
- 4,406 null values (0.0239%)

**2.4.3 YoJ (Year of Join):**
- 85 distinct values with complete sequential coverage 1941-2025
- No missing years in the range
- 39 null values (0.0002% - minimal missingness)
- Clean, consistent temporal data

**2.4.4 q_members_t:**
- 99 distinct values following mathematical pattern: base value (2.969121) × multipliers 1-100
- Missing multiplier: 99 (gap in sequence)
- Range: 2.97 to 296.91
- Perfect mathematical relationship confirms this is a calculated/weighted field

**2.4.5 q_leavers_t:**
- 11 distinct values using same base value (2.969121) × multipliers: 1-10, 17
- Missing multipliers: 11-16 (significant gap in sequence)
- Range: 2.97 to 50.48
- 18,259,005 null values (98.90%) - these represent members who are still active and have not left
- 202,475 non-null records (1.10%) - these represent members who have departed
- Strong concentration at the lowest value: 197,390 departed members (97.5% of all leavers) have the value 2.97, while the remaining 5,085 leavers are spread across the other 10 multiplier values
- This skewed distribution suggests that most members who leave belong to the same category or leave under similar circumstances, while only a small number fall into the higher multiplier groups

**Key Findings:**
- Both quantitative measures (q_members_t and q_leavers_t) are derived from the same base value
- The 98.90% null rate in q_leavers_t aligns with active membership expectations
- YoB contains data quality issues requiring correction (future birth years)
- Date and year columns show reliable patterns: CM_snapshot_date has perfect monthly coverage, and YoJ shows complete annual coverage from 1941-2025. YoB is mostly complete from 1917-2008 but contains some implausible values (future years like 2966) that indicate data entry errors.

**Status:** ⚠️ Investigate

### 2.5 Categorical Value Enumeration

**CONTEXT**

Categorical value enumeration examines the complete set of distinct values within string columns to understand classification schemes, identify data quality issues such as inconsistent formatting or unexpected entries, and validate the categorical structure of the dataset.

**PURPOSE**

To enumerate:
1. All distinct values for each categorical string column
2. Formatting inconsistencies (capitalization, spacing, duplicates)
3. The complete value set defining each categorical dimension

**STEP**

Extract and display all distinct values for categorical string columns in sorted order to examine naming conventions, identify inconsistencies, and understand the classification structure.

#### 2.5.1 Distinct Values - Int_nurse

In [0]:
# ═══════════════════════════════════════════════════════════════
# Int_nurse - Distinct Values & Counts
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import round as spark_round, sum as spark_sum, lit

print("Int_nurse - Distinct Values with Counts:")
print("-" * 70)

int_nurse_counts = df_raw.groupBy('Int_nurse').count() \
    .withColumn('percentage', spark_round((col('count') / total_rows) * 100, 2)) \
    .orderBy('count', ascending=False)

# Create totals row
totals_row = int_nurse_counts.select(
    lit('TOTAL').alias('Int_nurse'),
    spark_sum('count').alias('count'),
    spark_sum('percentage').alias('percentage')
)

# Union the data with totals
int_nurse_with_totals = int_nurse_counts.union(totals_row)

display(int_nurse_with_totals)

Int_nurse - Distinct Values with Counts:
----------------------------------------------------------------------


Int_nurse,count,percentage
UK,9827584,53.23
Other,5739784,31.09
Overseas,2214981,12.0
EEU,679131,3.68
TOTAL,18461480,100.0


**Relationship to Region - Int_nurse:**

Examine how the 4 Int_nurse values map to the 14 Region values to understand the geographic-international classification structure.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Int_nurse to Region Mapping
# ═══════════════════════════════════════════════════════════════

print("Int_nurse to Region MAPPING")
print("=" * 70)

# Get unique combinations of Int_nurse and Region
int_nurse_region_mapping = df_raw.select('Int_nurse', 'Region').distinct() \
    .orderBy('Int_nurse', 'Region')

print("\nComplete mapping (showing all Int_nurse-Region combinations):")
display(int_nurse_region_mapping)

# Count regions per Int_nurse
print("\n" + "=" * 70)
print("REGION COUNT BY Int_nurse")
print("=" * 70)

regions_per_int_nurse = int_nurse_region_mapping.groupBy('Int_nurse').count() \
    .orderBy('count', ascending=False) \
    .withColumnRenamed('count', 'num_regions')

display(regions_per_int_nurse)

# Detailed breakdown: each Int_nurse with its regions and counts
print("\n" + "=" * 70)
print("DETAILED Int_nurse-REGION BREAKDOWN WITH COUNTS")
print("=" * 70)

for int_nurse_val in int_nurse_region_mapping.select('Int_nurse').distinct().orderBy('Int_nurse').collect():
    int_nurse = int_nurse_val['Int_nurse']
    
    print(f"\n{int_nurse}:")
    print("-" * 70)
    
    # Get regions for this Int_nurse with counts
    int_nurse_regions = df_raw.filter(col('Int_nurse') == int_nurse) \
        .groupBy('Region').count() \
        .withColumn('percentage', spark_round((col('count') / total_rows) * 100, 2)) \
        .orderBy('count', ascending=False)
    
    display(int_nurse_regions)

Int_nurse to Region MAPPING

Complete mapping (showing all Int_nurse-Region combinations):


Int_nurse,Region
EEU,East Midlands
EEU,Eastern
EEU,H Q (Overseas)
EEU,London
EEU,North West
EEU,Northern
EEU,Northern Ireland
EEU,Scotland
EEU,South East
EEU,South West



REGION COUNT BY Int_nurse


Int_nurse,num_regions
Other,15
UK,13
EEU,13
Overseas,13



DETAILED Int_nurse-REGION BREAKDOWN WITH COUNTS

EEU:
----------------------------------------------------------------------


...

Region,count,percentage
South East,735377,3.98
London,697199,3.78
North West,615998,3.34
West Midlands,560693,3.04
Scotland,541135,2.93
South West,499588,2.71
Eastern,474465,2.57
East Midlands,408232,2.21
Yorkshire & The Humber,393336,2.13
Wales,378662,2.05



Overseas:
----------------------------------------------------------------------


Region,count,percentage
London,456281,2.47
South East,404743,2.19
Eastern,203417,1.1
West Midlands,203193,1.1
South West,196848,1.07
North West,192341,1.04
East Midlands,141164,0.76
Yorkshire & The Humber,115272,0.62
Wales,91700,0.5
Scotland,81854,0.44



UK:
----------------------------------------------------------------------


Region,count,percentage
South East,1379848,7.47
London,1101106,5.96
West Midlands,972126,5.27
South West,969089,5.25
Scotland,960239,5.2
North West,951639,5.15
Eastern,773915,4.19
East Midlands,693319,3.76
Yorkshire & The Humber,606989,3.29
Wales,578210,3.13


#### 2.5.2 Distinct Values - Region

In [0]:
# ═══════════════════════════════════════════════════════════════
# Region - Distinct Values & Counts
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import round as spark_round, sum as spark_sum, lit

print("Region - Distinct Values with Counts:")
print("-" * 70)

region_counts = df_raw.groupBy('Region').count() \
    .withColumn('percentage', spark_round((col('count') / total_rows) * 100, 2)) \
    .orderBy('count', ascending=False)

# Create totals row
totals_row = region_counts.select(
    lit('TOTAL').alias('Region'),
    spark_sum('count').alias('count'),
    spark_sum('percentage').alias('percentage')
)

# Union the data with totals
region_with_totals = region_counts.union(totals_row)

display(region_with_totals)

Region - Distinct Values with Counts:
----------------------------------------------------------------------


Region,count,percentage
South East,2675078,14.49
London,2406024,13.03
North West,1804964,9.78
West Midlands,1775766,9.62
South West,1741816,9.43
Scotland,1607809,8.71
Eastern,1524718,8.26
East Midlands,1277500,6.92
Yorkshire & The Humber,1142390,6.19
Wales,1067920,5.78


#### 2.5.3 Distinct Values - Branch

In [0]:
# ═══════════════════════════════════════════════════════════════
# Branch - Distinct Values & Counts
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import round as spark_round, sum as spark_sum, lit

print("Branch - Distinct Values with Counts (Top 20 and Bottom 20):")
print("-" * 70)

branch_counts = df_raw.groupBy('Branch').count() \
    .withColumn('percentage', spark_round((col('count') / total_rows) * 100, 2)) \
    .orderBy('count', ascending=False)

print("Top 20 branches by count:")
display(branch_counts.limit(20))

print("\nBottom 20 branches by count:")
bottom_20 = branch_counts.orderBy('count', ascending=True).limit(20)
display(bottom_20)

# Create totals row
totals_row = branch_counts.select(
    lit('TOTAL').alias('Branch'),
    spark_sum('count').alias('count'),
    spark_sum('percentage').alias('percentage')
)

print("\nTotal:")
display(totals_row)

Branch - Distinct Values with Counts (Top 20 and Bottom 20):
----------------------------------------------------------------------
Top 20 branches by count:


Branch,count,percentage
West Yorkshire,400031,2.17
Essex,365811,1.98
South East London Inner,322667,1.75
Manchester Central,315711,1.71
South Yorkshire,309879,1.68
Surrey,298107,1.61
Greater Bristol,297783,1.61
Lancashire West,294187,1.59
Hampshire,286701,1.55
Northumberland Tyne and Wear,284474,1.54



Bottom 20 branches by count:


Branch,count,percentage
NON MEMBERS,1,0.0
Not Provided,1,0.0
RCN HQ Staff Centre,1054,0.01
Shetland,10799,0.06
Western Isles,21899,0.12
Overseas Branch,24612,0.13
Baliwick of Guernsey,42922,0.23
Isle of Man,45684,0.25
South Western,49227,0.27
Jersey,49521,0.27



Total:


Branch,count,percentage
TOTAL,18461480,99.96


**Relationship to Region - Branch:**

Examine how the 104 Branch values map to the 14 Region values to understand the geographic hierarchy.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Branch to Region Mapping
# ═══════════════════════════════════════════════════════════════

print("Branch to Region MAPPING")
print("=" * 70)

# Get unique combinations of Region and Branch
region_branch_mapping = df_raw.select('Region', 'Branch').distinct() \
    .orderBy('Region', 'Branch')

print("\nComplete mapping (showing all Region-Branch combinations):")
display(region_branch_mapping)

# Count branches per region
print("\n" + "=" * 70)
print("BRANCH COUNT BY REGION")
print("=" * 70)

branches_per_region = region_branch_mapping.groupBy('Region').count() \
    .orderBy('count', ascending=False) \
    .withColumnRenamed('count', 'num_branches')

display(branches_per_region)

# Detailed breakdown: each region with its branches and counts
print("\n" + "=" * 70)
print("DETAILED REGION-BRANCH BREAKDOWN WITH COUNTS")
print("=" * 70)

for region_name in region_branch_mapping.select('Region').distinct().orderBy('Region').collect():
    region = region_name['Region']
    
    print(f"\n{region}:")
    print("-" * 70)
    
    # Get branches for this region with counts
    region_branches = df_raw.filter(col('Region') == region) \
        .groupBy('Branch').count() \
        .withColumn('percentage', spark_round((col('count') / total_rows) * 100, 2)) \
        .orderBy('count', ascending=False)
    
    display(region_branches)

Branch to Region MAPPING

Complete mapping (showing all Region-Branch combinations):


Region,Branch
null,Not Provided
East Midlands,Derbyshire
East Midlands,Leicestershire & Rutland
East Midlands,North Lincolnshire
East Midlands,North Nottinghamshire
East Midlands,Northamptonshire
East Midlands,Nottingham
East Midlands,Nottinghamshire
East Midlands,South Lincolnshire
Eastern,Bedfordshire



BRANCH COUNT BY REGION


Region,num_branches
South East,15
Scotland,13
London,11
West Midlands,10
South West,9
North West,9
East Midlands,8
Northern Ireland,6
Eastern,6
Wales,6



DETAILED REGION-BRANCH BREAKDOWN WITH COUNTS

None:
----------------------------------------------------------------------


...

Branch,count,percentage
Cangen Gogledd Cymru,218382,1.18
Cardiff and the Vale,196154,1.06
Cwm Taf Morgannwg,175661,0.95
Glamorgan,174873,0.95
Gwent,163654,0.89
Mid and West Wales,139196,0.75



West Midlands:
----------------------------------------------------------------------


Branch,count,percentage
Coventry and Warwickshire,237348,1.29
Black Country,219785,1.19
South Birmingham,216775,1.17
Birmingham E N and Solihull,188398,1.02
Worcestershire,187984,1.02
South Staffordshire,183876,1.0
North Staffordshire,172597,0.93
Birmingham West and Sandwell,153678,0.83
Shropshire,141217,0.76
Herefordshire,74108,0.4



Yorkshire & The Humber:
----------------------------------------------------------------------


Branch,count,percentage
West Yorkshire,400031,2.17
South Yorkshire,309879,1.68
Humber,219596,1.19
North Yorkshire,212884,1.15


#### 2.5.4 Distinct Values - MemCategory

In [0]:
# ═══════════════════════════════════════════════════════════════
# MemCategory - Distinct Values & Counts
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import round as spark_round, sum as spark_sum, lit

print("MemCategory - Distinct Values with Counts:")
print("-" * 70)

memcategory_counts = df_raw.groupBy('MemCategory').count() \
    .withColumn('percentage', spark_round((col('count') / total_rows) * 100, 2)) \
    .orderBy('count', ascending=False)

# Create totals row
totals_row = memcategory_counts.select(
    lit('TOTAL').alias('MemCategory'),
    spark_sum('count').alias('count'),
    spark_sum('percentage').alias('percentage')
)

# Union the data with totals
memcategory_with_totals = memcategory_counts.union(totals_row)

display(memcategory_with_totals)

MemCategory - Distinct Values with Counts:
----------------------------------------------------------------------


MemCategory,count,percentage
Nurse member,16022515,86.79
Nursing Support Worker,1054250,5.71
Student,709228,3.84
Nurse Support Worker,675487,3.66
TOTAL,18461480,100.0


**Pattern Investigation - MemCategory:**

Examine potential duplicate categories with similar naming (Nursing Support Worker vs Nurse Support Worker).

In [0]:
# ═══════════════════════════════════════════════════════════════
# MemCategory - Duplicate Category Investigation
# ═══════════════════════════════════════════════════════════════

print("NURSING/NURSE SUPPORT WORKER OVERLAP ANALYSIS")
print("=" * 70)

# Check if any records have both categories in different snapshots
# Group by all non-temporal dimensions to see if same cohorts have different MemCategory labels

print("\nChecking for cohort overlap between the two support worker categories:")
print("-" * 70)

# Check if there are records with identical dimensions except MemCategory
nsw_records = df_raw.filter(col('MemCategory') == 'Nurse Support Worker') \
    .select('Int_nurse', 'Region', 'Branch', 'YoB', 'YoJ', 'MemSectorType').distinct()

nursing_sw_records = df_raw.filter(col('MemCategory') == 'Nursing Support Worker') \
    .select('Int_nurse', 'Region', 'Branch', 'YoB', 'YoJ', 'MemSectorType').distinct()

# Check for intersection
overlap = nsw_records.intersect(nursing_sw_records)
overlap_count = overlap.count()

print(f"Unique cohort combinations in 'Nurse Support Worker': {nsw_records.count():,}")
print(f"Unique cohort combinations in 'Nursing Support Worker': {nursing_sw_records.count():,}")
print(f"Overlapping cohort combinations: {overlap_count:,}")

if overlap_count > 0:
    print("\n⚠️ OVERLAP DETECTED: Same cohorts appear under both category names")
    print("This suggests inconsistent labeling rather than distinct categories")
else:
    print("\n✓ NO OVERLAP: These appear to be distinct member populations")

# Check temporal distribution
print("\n" + "=" * 70)
print("TEMPORAL DISTRIBUTION")
print("=" * 70)

temporal_dist = df_raw.filter(
    (col('MemCategory') == 'Nurse Support Worker') | 
    (col('MemCategory') == 'Nursing Support Worker')
).groupBy('CM_snapshot_date', 'MemCategory').count().orderBy('CM_snapshot_date', 'MemCategory')

print("\nSample of temporal distribution (first 20 rows):")
display(temporal_dist.limit(20))

# Check if categories appear in different time periods
print("\n" + "=" * 70)
print("CATEGORY APPEARANCE BY TIME PERIOD")
print("=" * 70)

nsw_dates = df_raw.filter(col('MemCategory') == 'Nurse Support Worker') \
    .select('CM_snapshot_date').distinct().orderBy('CM_snapshot_date')

nursing_sw_dates = df_raw.filter(col('MemCategory') == 'Nursing Support Worker') \
    .select('CM_snapshot_date').distinct().orderBy('CM_snapshot_date')

print("\nNurse Support Worker appears in:")
nsw_date_list = [row['CM_snapshot_date'] for row in nsw_dates.collect()]
print(f"  Date range: {nsw_date_list[0]} to {nsw_date_list[-1]}")
print(f"  Number of months: {len(nsw_date_list)}")

print("\nNursing Support Worker appears in:")
nursing_sw_date_list = [row['CM_snapshot_date'] for row in nursing_sw_dates.collect()]
print(f"  Date range: {nursing_sw_date_list[0]} to {nursing_sw_date_list[-1]}")
print(f"  Number of months: {len(nursing_sw_date_list)}")

# Check if they overlap temporally
overlap_dates = set(nsw_date_list).intersection(set(nursing_sw_date_list))
print(f"\nMonths where both categories appear: {len(overlap_dates)}")

if len(overlap_dates) == 0:
    print("✓ Categories appear in different time periods - likely a naming change/correction")
else:
    print("⚠️ Categories coexist in same time periods - likely inconsistent labeling")

NURSING/NURSE SUPPORT WORKER OVERLAP ANALYSIS

Checking for cohort overlap between the two support worker categories:
----------------------------------------------------------------------
Unique cohort combinations in 'Nurse Support Worker': 44,263
Unique cohort combinations in 'Nursing Support Worker': 48,488
Overlapping cohort combinations: 29,879

⚠️ OVERLAP DETECTED: Same cohorts appear under both category names
This suggests inconsistent labeling rather than distinct categories

TEMPORAL DISTRIBUTION

Sample of temporal distribution (first 20 rows):


CM_snapshot_date,MemCategory,count
2021-01-01,Nursing Support Worker,22232
2021-02-01,Nursing Support Worker,22695
2021-03-01,Nursing Support Worker,23026
2021-04-01,Nursing Support Worker,23595
2021-05-01,Nursing Support Worker,23851
2021-06-01,Nursing Support Worker,24024
2021-07-01,Nursing Support Worker,24202
2021-08-01,Nursing Support Worker,24244
2021-09-01,Nursing Support Worker,24110
2021-10-01,Nursing Support Worker,24190



CATEGORY APPEARANCE BY TIME PERIOD

Nurse Support Worker appears in:
  Date range: 2024-05-01 to 2025-12-01
  Number of months: 20

Nursing Support Worker appears in:
  Date range: 2021-01-01 to 2024-04-01
  Number of months: 40

Months where both categories appear: 0
✓ Categories appear in different time periods - likely a naming change/correction


#### 2.5.5 Distinct Values - CatName

In [0]:
# ═══════════════════════════════════════════════════════════════
# CatName - Distinct Values & Counts
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import round as spark_round, sum as spark_sum, lit

print("CatName - Distinct Values with Counts (Alphabetically):")
print("-" * 70)

catname_counts = df_raw.groupBy('CatName').count() \
    .withColumn('percentage', spark_round((col('count') / total_rows) * 100, 2)) \
    .orderBy('CatName')

# Create totals row
totals_row = catname_counts.select(
    lit('TOTAL').alias('CatName'),
    spark_sum('count').alias('count'),
    spark_sum('percentage').alias('percentage')
)

# Union the data with totals
catname_with_totals = catname_counts.union(totals_row)

display(catname_with_totals)

CatName - Distinct Values with Counts (Alphabetically):
----------------------------------------------------------------------


CatName,count,percentage
Joint - RCN/EIS,3099,0.02
Joint - RCN/RCM,30458,0.16
Joint - RCN/UCU,43615,0.24
Life Member,30864,0.17
Nurse - 1st Year Discount,852561,4.62
Nurse - Career Break,126003,0.68
Nurse - RCN Staff,2686,0.01
Nurse - Retired,808679,4.38
Nurse - Voluntary Break,2455,0.01
Nurse Full,14122095,76.49


**Relationship to MemCategory - CatName:**

Examine how the 17 CatName values map to the 4 MemCategory values to understand the hierarchical classification structure.

In [0]:
# ═══════════════════════════════════════════════════════════════
# CatName to MemCategory Mapping
# ═══════════════════════════════════════════════════════════════

print("CatName to MemCategory MAPPING")
print("=" * 70)

# Get unique combinations of CatName and MemCategory
mapping = df_raw.select('MemCategory', 'CatName').distinct() \
    .orderBy('MemCategory', 'CatName')

print("\nComplete mapping:")
display(mapping)

# Count CatName values per MemCategory
print("\n" + "=" * 70)
print("CatName COUNT BY MemCategory")
print("=" * 70)

catname_per_memcat = mapping.groupBy('MemCategory').count() \
    .orderBy('count', ascending=False) \
    .withColumnRenamed('count', 'num_catname_values')

display(catname_per_memcat)

CatName to MemCategory MAPPING

Complete mapping:


MemCategory,CatName
Nurse Support Worker,Nursing Support Worker - 1st Year Discount
Nurse Support Worker,Nursing Support Worker - Retired
Nurse Support Worker,Nursing Support Worker - Student Nursing Associate
Nurse Support Worker,Nursing Support Worker -Career Break
Nurse Support Worker,Nursing Support Worker Full
Nurse member,Joint - RCN/EIS
Nurse member,Joint - RCN/RCM
Nurse member,Joint - RCN/UCU
Nurse member,Life Member
Nurse member,Nurse - 1st Year Discount



CatName COUNT BY MemCategory


MemCategory,num_catname_values
Nurse member,10
Nursing Support Worker,5
Nurse Support Worker,5
Student,1


#### 2.5.6 Distinct Values - MemSectorType

In [0]:
# ═══════════════════════════════════════════════════════════════
# MemSectorType - Distinct Values & Counts
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import round as spark_round, sum as spark_sum, lit

print("MemSectorType - Distinct Values with Counts:")
print("-" * 70)

memsectortype_counts = df_raw.groupBy('MemSectorType').count() \
    .withColumn('percentage', spark_round((col('count') / total_rows) * 100, 2)) \
    .orderBy('count', ascending=False)

# Create totals row
totals_row = memsectortype_counts.select(
    lit('TOTAL').alias('MemSectorType'),
    spark_sum('count').alias('count'),
    spark_sum('percentage').alias('percentage')
)

# Union the data with totals
memsectortype_with_totals = memsectortype_counts.union(totals_row)

display(memsectortype_with_totals)

MemSectorType - Distinct Values with Counts:
----------------------------------------------------------------------


MemSectorType,count,percentage
NHS,10024494,54.3
Independent,4981521,26.98
Education,1711549,9.27
null,1436695,7.78
Other public sector,267728,1.45
Other Public Sector,39493,0.21
TOTAL,18461480,99.99


**Data Quality & Relationship Analysis - MemSectorType:**

Identify casing inconsistencies and examine how MemSectorType maps to other categorical dimensions.

In [0]:
# ═══════════════════════════════════════════════════════════════
# MemSectorType - Casing Issue & Mapping Analysis
# ═══════════════════════════════════════════════════════════════

print("CASING INCONSISTENCY VERIFICATION")
print("=" * 70)

# Confirm the casing issue
other_public_lower = df_raw.filter(col('MemSectorType') == 'Other public sector').count()
other_public_title = df_raw.filter(col('MemSectorType') == 'Other Public Sector').count()
combined_total = other_public_lower + other_public_title

print(f"'Other public sector' (lowercase): {other_public_lower:,}")
print(f"'Other Public Sector' (title case): {other_public_title:,}")
print(f"Combined total: {combined_total:,}")
print(f"Combined percentage: {(combined_total / total_rows) * 100:.2f}%")

# Mapping to MemCategory
print("\n" + "=" * 70)
print("MemSectorType TO MemCategory MAPPING")
print("=" * 70)

sector_memcat_mapping = df_raw.groupBy('MemSectorType', 'MemCategory').count() \
    .orderBy('MemSectorType', 'count', ascending=[True, False])

display(sector_memcat_mapping)

# Check where nulls appear
print("\n" + "=" * 70)
print("NULL MemSectorType DISTRIBUTION")
print("=" * 70)

null_distribution = df_raw.filter(col('MemSectorType').isNull()) \
    .groupBy('MemCategory').count() \
    .withColumn('percentage_of_nulls', spark_round((col('count') / 1436695) * 100, 2)) \
    .orderBy('count', ascending=False)

print("Which MemCategory values have null MemSectorType:")
display(null_distribution)

# Alternative view: By MemCategory
print("\n" + "=" * 70)
print("MemCategory TO MemSectorType MAPPING (Clearer View)")
print("=" * 70)

for memcat in ['Nurse member', 'Nursing Support Worker', 'Nurse Support Worker', 'Student']:
    print(f"\n{memcat}:")
    print("-" * 70)
    
    memcat_sectors = df_raw.filter(col('MemCategory') == memcat) \
        .groupBy('MemSectorType').count() \
        .withColumn('percentage', spark_round((col('count') / total_rows) * 100, 2)) \
        .orderBy('count', ascending=False)
    
    display(memcat_sectors)

CASING INCONSISTENCY VERIFICATION
'Other public sector' (lowercase): 267,728
'Other Public Sector' (title case): 39,493
Combined total: 307,221
Combined percentage: 1.66%

MemSectorType TO MemCategory MAPPING


MemSectorType,MemCategory,count
null,Nurse member,1372081
null,Nursing Support Worker,29426
null,Nurse Support Worker,19841
null,Student,15347
Education,Nurse member,1053328
Education,Student,644799
Education,Nursing Support Worker,8232
Education,Nurse Support Worker,5190
Independent,Nurse member,4408002
Independent,Nursing Support Worker,333876



NULL MemSectorType DISTRIBUTION
Which MemCategory values have null MemSectorType:


MemCategory,count,percentage_of_nulls
Nurse member,1372081,95.5
Nursing Support Worker,29426,2.05
Nurse Support Worker,19841,1.38
Student,15347,1.07



MemCategory TO MemSectorType MAPPING (Clearer View)

Nurse member:
----------------------------------------------------------------------


...

MemSectorType,count,percentage
NHS,667277,3.61
Independent,333876,1.81
null,29426,0.16
Other public sector,13770,0.07
Education,8232,0.04
Other Public Sector,1669,0.01



Nurse Support Worker:
----------------------------------------------------------------------


MemSectorType,count,percentage
NHS,416916,2.26
Independent,224353,1.22
null,19841,0.11
Other public sector,8615,0.05
Education,5190,0.03
Other Public Sector,572,0.0



Student:
----------------------------------------------------------------------


MemSectorType,count,percentage
Education,644799,3.49
NHS,32580,0.18
null,15347,0.08
Independent,15290,0.08
Other public sector,1166,0.01
Other Public Sector,46,0.0


**RESULT**

Categorical value enumeration reveals hierarchical geographic and membership classification structures with notable data quality issues:

**2.5.1 Int_nurse (International Classification):**
- 4 distinct values: UK (9,827,584 records, 53.23%), Other (5,739,784, 31.09%), Overseas (2,214,981, 12.00%), EEU (679,131, 3.68%)
- All four categories appear across all 13 UK geographic regions, indicating this field does not represent member location
- UK members dominate in every region including overseas headquarters
- "Other" category contains two data anomalies: one null record and one "Non Members" record
- Distribution pattern suggests Int_nurse likely represents qualification or registration origin rather than current geographic location

**2.5.2 Region (Geographic Classification):**
- 14 distinct values plus one null record and one "Non Members" record
- Largest regions: South East (2,675,078, 14.49%), London (2,406,024, 13.03%), North West (1,804,964, 9.78%)
- Smallest regions: Northern Ireland (684,325, 3.71%), H Q (Overseas) (24,612, 0.13%)
- Two anomalous single-record entries: null and "Non Members"
- Maps to 104 branches in hierarchical structure

**2.5.3 Branch (Local Geographic Units):**
- 104 distinct branches mapping to 14 regions in clear hierarchical relationship
- Branch distribution per region: South East (15 branches), Scotland (13), London (11), West Midlands (10), South West and North West (9 each), East Midlands (8), Northern Ireland, Eastern, and Wales (6 each), Northern and Yorkshire & The Humber (4 each), H Q Overseas (1), null (1), Non Members (1)
- Largest branches: West Yorkshire (400,031, 2.17%), Essex (365,811, 1.98%), South East London Inner (322,667, 1.75%)
- Smallest branches: the organisation HQ Staff Centre (1,054, 0.01%), with two single-record anomalies ("Not Provided" and "NON MEMBERS")
- Three branches associated with data quality issues: "Overseas Branch" maps to H Q (Overseas) region, while "Not Provided" and "NON MEMBERS" appear as anomalies
- Regional breakdown confirms branches are sub-units of regions with detailed member distribution available

**2.5.4 MemCategory (Membership Type - Broad Classification):**
- 4 distinct values representing primary membership classifications
- Nurse member: 16,022,515 (86.79%) - dominant category
- Nursing Support Worker: 1,054,250 (5.71%) - appears only 2021-01-01 to 2024-04-01
- Nurse Support Worker: 675,487 (3.66%) - appears only 2024-05-01 to 2025-12-01 
- Student: 709,228 (3.84%)
- **Critical finding**: "Nursing Support Worker" and "Nurse Support Worker" represent the same category with a naming change implemented in May 2024. No temporal overlap exists between these labels, confirming this is a systematic renaming rather than inconsistent data entry.
- Combined support worker total: 1,729,737 (9.37%)
- Effectively 3 distinct membership types: Nurse members, Support workers, Students

**2.5.5 CatName (Membership Type - Detailed Classification):**
- 17 distinct values providing granular membership subcategories
- Maps hierarchically to MemCategory: Nurse member (10 subcategories), Nursing Support Worker (5 subcategories), Nurse Support Worker (5 subcategories), Student (1 subcategory)
- Largest subcategories: Nurse Full (14,122,095, 76.49%), Nursing Support Worker Full (1,314,386, 7.12%), Nurse - 1st Year Discount (852,561, 4.62%)
- Subcategories reveal membership structure: Full membership, discounted first year, retired status, career breaks, voluntary breaks, joint memberships (the organisation/UCU, the organisation/RCM, the organisation/EIS), life members, the organisation staff, and student nursing associates
- Joint memberships represent 77,172 records (0.42%) across three union partnerships
- Specialized categories exist for both nurse and support worker classifications (retired, career break, voluntary break, first year discount)

**2.5.6 MemSectorType (Employment Sector):**
- 5 distinct values (treating casing variants as one) plus 1,436,695 null values (7.78%)
- NHS: 10,024,494 (54.30%) - largest sector
- Independent: 4,981,521 (26.98%)
- Education: 1,711,549 (9.27%)
- Other public sector: 307,221 (1.66%) - **Data quality issue**: appears as both "Other public sector" (267,728) and "Other Public Sector" (39,493) due to inconsistent capitalization
- All sector types appear across all four MemCategory classifications
- Null distribution heavily skewed toward Nurse members: 1,372,081 nulls (95.5% of all MemSectorType nulls) are Nurse members, with remaining 4.5% distributed across support workers (3.43%) and students (1.07%)
- High null rate (7.78%) suggests this field may be optional, inconsistently collected, or added later in data collection timeline

**Data Quality Issues Identified:**
1. Casing inconsistency in MemSectorType ("Other public sector" vs "Other Public Sector")
2. Naming convention change in MemCategory (Nursing → Nurse Support Worker in May 2024)
3. Anomalous single-record entries across multiple columns (null, "Non Members", "Not Provided")
4. Substantial missingness in MemSectorType (7.78%)
5. "Other" category in Int_nurse serves as catch-all including data anomalies

**Key Findings:**
- Clear two-tier geographic hierarchy for work location: Region (14 areas) → Branch (104 local units), with branches serving as sub-units within regions
- Int_nurse classification (4 categories) operates independently of geographic location, as all categories appear across all UK regions. The distribution pattern and naming suggest this likely represents qualification or registration origin rather than current work location - these are distinct location captures serving different purposes
- Clear two-tier membership classification hierarchy: MemCategory (4 broad types, effectively 3 after accounting for naming change) → CatName (17 detailed subcategories)
- MemSectorType shows independence from membership classifications, appearing across all member types, and likely represents employer sector rather than member characteristics
- Temporal analysis confirms a category renaming in May 2024: "Nursing Support Worker" (used 2021-2024) was replaced by "Nurse Support Worker" (used 2024-2025) with no overlap between the two labels, indicating a clean systematic change rather than inconsistent data entry
- High concentration in specific categories: 86.79% are Nurse members, 54.30% work in NHS
- Data quality issues are isolated and documentable, primarily affecting edge cases and null handling

**Status:** ⚠️ Investigate

### 2.6 Logical Consistency Checks

**CONTEXT**

Logical consistency checks validate that relationships between variables conform to expected business rules and constraints. This analysis examines whether data values maintain logical coherence across related columns, such as ensuring departure counts do not exceed membership counts or that temporal sequences align properly.

**PURPOSE**

To verify:
1. q_leavers_t values do not exceed q_members_t values (departures cannot exceed total members)
2. Temporal relationships are logically sound (join years do not precede birth years by implausible margins)
3. Category combinations are valid (e.g., Students in Education sector)
4. Hierarchical relationships maintain integrity (Branch values align with their Region)

**STEP**

Perform cross-column validations to identify records violating logical constraints, quantify the extent of violations, and assess the integrity of hierarchical and temporal relationships. This analysis comprises four targeted checks: leaver/member constraint validation, birth/join temporal logic, student/sector alignment, and branch/region hierarchical integrity.

#### 2.6.1 Leaver/Member Constraint Validation

In [0]:
# ═══════════════════════════════════════════════════════════════
# Check 1: q_leavers_t should never exceed q_members_t
# ═══════════════════════════════════════════════════════════════

print("LEAVER/MEMBER RELATIONSHIP CHECK")
print("=" * 70)

violations_leavers = df_raw.filter(
    (col('q_leavers_t').isNotNull()) & 
    (col('q_leavers_t') > col('q_members_t'))
).count()

print(f"Records where q_leavers_t > q_members_t: {violations_leavers:,}")
if violations_leavers == 0:
    print("✓ PASS: No violations found - departure counts never exceed membership counts")
else:
    print("⚠️ VIOLATION: Leavers exceed members in some records")
    # Show examples
    examples = df_raw.filter(
        (col('q_leavers_t').isNotNull()) & 
        (col('q_leavers_t') > col('q_members_t'))
    ).select('q_members_t', 'q_leavers_t', 'CM_snapshot_date', 'Region', 'MemCategory').limit(10)
    display(examples)

LEAVER/MEMBER RELATIONSHIP CHECK
Records where q_leavers_t > q_members_t: 0
✓ PASS: No violations found - departure counts never exceed membership counts


#### 2.6.2 Birth/Join Temporal Logic Validation

In [0]:
# ═══════════════════════════════════════════════════════════════
# Check 2: Birth/Join Temporal Logic
# ═══════════════════════════════════════════════════════════════

import builtins

print("BIRTH/JOIN TEMPORAL LOGIC CHECK")
print("=" * 70)

# Check 2a: Minimum joining age (assuming 16 years minimum)
print("\n2a. Minimum Join Age Validation:")
print("-" * 70)

min_join_age = 16

age_violations = df_raw.filter(
    (col('YoB').isNotNull()) & 
    (col('YoJ').isNotNull()) & 
    ((col('YoJ') - col('YoB')) < min_join_age)
).count()

print(f"Records where join age < {min_join_age} years: {age_violations:,}")
print(f"Percentage: {(age_violations / total_rows) * 100:.4f}%")

if age_violations > 0:
    print("\nSample violations:")
    examples = df_raw.filter(
        (col('YoB').isNotNull()) & 
        (col('YoJ').isNotNull()) & 
        ((col('YoJ') - col('YoB')) < min_join_age)
    ).select('YoB', 'YoJ', (col('YoJ') - col('YoB')).alias('join_age')) \
     .orderBy('join_age').limit(10)
    display(examples)

# Check 2b: Future birth years and negative ages
print("\n2b. Future Birth Year & Negative Age Check:")
print("-" * 70)

current_year = 2026  # Based on current date from context

future_births = df_raw.filter(
    (col('YoB').isNotNull()) & 
    (col('YoB') > current_year)
).count()

negative_age_joins = df_raw.filter(
    (col('YoB').isNotNull()) & 
    (col('YoJ').isNotNull()) & 
    ((col('YoJ') - col('YoB')) < 0)
).count()

print(f"Records with birth year > {current_year}: {future_births:,}")
print(f"Records with negative join age (joined before birth): {negative_age_joins:,}")

if future_births > 0:
    print("\nFuture birth year examples:")
    future_examples = df_raw.filter(col('YoB') > current_year) \
        .select('YoB', 'YoJ', 'MemCategory', 'Region') \
        .distinct().orderBy('YoB')
    display(future_examples)

if negative_age_joins > 0:
    print("\nNegative age examples:")
    negative_examples = df_raw.filter(
        (col('YoB').isNotNull()) & 
        (col('YoJ').isNotNull()) & 
        ((col('YoJ') - col('YoB')) < 0)
    ).select('YoB', 'YoJ', (col('YoJ') - col('YoB')).alias('join_age')) \
     .orderBy('join_age').limit(10)
    display(negative_examples)

BIRTH/JOIN TEMPORAL LOGIC CHECK

2a. Minimum Join Age Validation:
----------------------------------------------------------------------
Records where join age < 16 years: 1,183
Percentage: 0.0064%

Sample violations:


YoB,YoJ,join_age
2966.0,2024.0,-942.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2044.0,1985.0,-59.0
2042.0,2023.0,-19.0



2b. Future Birth Year & Negative Age Check:
----------------------------------------------------------------------
Records with birth year > 2026: 10
Records with negative join age (joined before birth): 34

Future birth year examples:


YoB,YoJ,MemCategory,Region
2029.0,1969.0,Nurse member,North West
2042.0,2023.0,Nurse member,South West
2044.0,1985.0,Nurse member,H Q (Overseas)
2966.0,2024.0,Nurse member,Northern



Negative age examples:


YoB,YoJ,join_age
2966.0,2024.0,-942.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2029.0,1969.0,-60.0
2044.0,1985.0,-59.0
2042.0,2023.0,-19.0


#### 2.6.3 Student/Sector Alignment Validation

In [0]:
# ═══════════════════════════════════════════════════════════════
# Check 3: Student/Education Sector Alignment
# ═══════════════════════════════════════════════════════════════

print("STUDENT/EDUCATION SECTOR ALIGNMENT")
print("=" * 70)

students_total = df_raw.filter(col('MemCategory') == 'Student').count()
students_in_education = df_raw.filter(
    (col('MemCategory') == 'Student') & 
    (col('MemSectorType') == 'Education')
).count()
students_null_sector = df_raw.filter(
    (col('MemCategory') == 'Student') & 
    (col('MemSectorType').isNull())
).count()

print(f"Total Students: {students_total:,}")
print(f"Students in Education sector: {students_in_education:,} ({(students_in_education/students_total)*100:.2f}%)")
print(f"Students with null MemSectorType: {students_null_sector:,} ({(students_null_sector/students_total)*100:.2f}%)")

# Show distribution of students across sectors
print("\nStudent Distribution Across Sectors:")
print("-" * 70)

student_sectors = df_raw.filter(col('MemCategory') == 'Student') \
    .groupBy('MemSectorType').count() \
    .withColumn('percentage', spark_round((col('count') / students_total) * 100, 2)) \
    .orderBy('count', ascending=False)
    
display(student_sectors)

# Calculate students in clinical/work settings (NHS + Independent)
students_in_clinical = df_raw.filter(
    (col('MemCategory') == 'Student') & 
    (col('MemSectorType').isin(['NHS', 'Independent']))
).count()

print(f"\nStudents in clinical/work settings (NHS + Independent): {students_in_clinical:,} ({(students_in_clinical/students_total)*100:.2f}%)")
print("✓ Pattern consistent with placement/clinical training requirements")

STUDENT/EDUCATION SECTOR ALIGNMENT
Total Students: 709,228
Students in Education sector: 644,799 (90.92%)
Students with null MemSectorType: 15,347 (2.16%)

Student Distribution Across Sectors:
----------------------------------------------------------------------


MemSectorType,count,percentage
Education,644799,90.92
NHS,32580,4.59
null,15347,2.16
Independent,15290,2.16
Other public sector,1166,0.16
Other Public Sector,46,0.01



Students in clinical/work settings (NHS + Independent): 47,870 (6.75%)
✓ Pattern consistent with placement/clinical training requirements


#### 2.6.4 Branch/Region Hierarchical Integrity Validation

In [0]:
# ═══════════════════════════════════════════════════════════════
# Check 4: Branch-Region Hierarchical Integrity
# ═══════════════════════════════════════════════════════════════

print("BRANCH-REGION HIERARCHICAL INTEGRITY")
print("=" * 70)

# Check if any Branch appears in multiple Regions (should not happen in proper hierarchy)
branch_region_combos = df_raw.select('Branch', 'Region').distinct()
branches_with_multiple_regions = branch_region_combos.groupBy('Branch').count() \
    .filter(col('count') > 1)

multi_region_branches = branches_with_multiple_regions.count()

print(f"Branches appearing in multiple Regions: {multi_region_branches}")

if multi_region_branches == 0:
    print("✓ PASS: Each branch belongs to exactly one region")
    print("✓ Hierarchical integrity maintained - no orphaned or duplicated branches")
else:
    print("⚠️ VIOLATION: Some branches span multiple regions")
    print("\nBranches with multiple region assignments:")
    display(branches_with_multiple_regions)
    
    # Show examples
    print("\nExample violations:")
    for row in branches_with_multiple_regions.limit(5).collect():
        branch_name = row['Branch']
        regions = df_raw.filter(col('Branch') == branch_name).select('Region').distinct()
        print(f"\n{branch_name} appears in:")
        display(regions)

# Summary statistics
total_branches = df_raw.select('Branch').distinct().count()
total_regions = df_raw.select('Region').distinct().count()
print(f"\nHierarchy Summary:")
print("-" * 70)
print(f"Total unique Branches: {total_branches}")
print(f"Total unique Regions: {total_regions}")
print(f"Average branches per region: {total_branches / total_regions:.1f}")

BRANCH-REGION HIERARCHICAL INTEGRITY
Branches appearing in multiple Regions: 0
✓ PASS: Each branch belongs to exactly one region
✓ Hierarchical integrity maintained - no orphaned or duplicated branches

Hierarchy Summary:
----------------------------------------------------------------------
Total unique Branches: 104
Total unique Regions: 15
Average branches per region: 6.9


**RESULT**

Logical consistency validation across four targeted checks reveals strong data integrity with isolated temporal anomalies:

**2.6.1 Leaver/Member Constraint:**
- Zero violations detected across all 18,461,480 records
- No instances where q_leavers_t exceeds q_members_t
- Confirms mathematical integrity of departure versus membership counts

**2.6.2 Birth/Join Temporal Logic:**
- 1,183 records (0.0064%) show join age below 16 years minimum
- 10 records contain future birth years (2029, 2042, 2044, 2966)
- 34 records show negative join ages (joined before birth)
- Most severe violation: YoB 2966 with join year 2024 (join age -942)
- Pattern analysis suggests data entry errors from legacy two-digit year systems: values like YoB 2029 with YoJ 1969 (join age -60) likely represent YoB 1929, where early database systems using two-digit years were incorrectly converted to four-digit format in the 2000s
- Single outlier YoB 2966 appears to be input error, possibly intended as 1966 or 1996
- Violations represent 0.0064% of dataset - minimal but require correction

**2.6.3 Student/Sector Alignment:**
- 709,228 total students across dataset
- 644,799 students (90.92%) correctly aligned with Education sector
- 47,870 students (6.75%) in clinical/work settings (NHS: 32,580, Independent: 15,290)
- 15,347 students (2.16%) with null MemSectorType
- Distribution pattern consistent with UK nursing education requirements where students complete clinical placements in NHS/Independent settings while enrolled in educational programs
- Small proportion in "Other public sector" (1,212 combined, 0.17%) represents specialized training contexts
- Alignment validates sector classification logic

**2.6.4 Branch/Region Hierarchical Integrity:**
- Zero violations: each of 104 branches maps to exactly one region
- No orphaned branches or multi-region assignments
- 15 unique regions with average of 6.9 branches per region
- Clean hierarchical structure with no integrity issues
- Geographic hierarchy fully maintained

**Key Findings:**
- Three of four validation checks show complete data integrity (leaver/member constraint, student/sector alignment, branch/region hierarchy)
- Temporal logic violations are minimal (0.0064%) and attributable to identifiable data entry errors from legacy system conversions
- Student distribution across sectors aligns with expected clinical training patterns
- No structural integrity issues detected in hierarchical or quantitative relationships

**Status:** ✓ Pass (with minor data quality issues documented)

### 2.7 Temporal Consistency

**CONTEXT**

Temporal consistency analysis examines time-series patterns within the snapshot data to identify trends, seasonal variations, and anomalies across the 60-month observation period. This validation ensures temporal stability and detects unexpected shifts in membership patterns.

**PURPOSE**

To assess:
1. Membership trends over time (growth, decline, stability)
2. Consistency of snapshot counts across temporal periods
3. Temporal distribution of categorical variables (region, sector, category)
4. Identification of unusual spikes or drops in membership counts

**STEP**

Aggregate membership data by snapshot date to examine temporal trends, calculate month-over-month changes, and assess the stability of categorical distributions across the five-year observation period.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Temporal Consistency Analysis
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import sum as spark_sum, lag, year, month
from pyspark.sql.window import Window

print("TEMPORAL CONSISTENCY ANALYSIS")
print("=" * 70)

# Aggregate total records by snapshot date
print("\n1. Total Records by Snapshot Date:")
print("-" * 70)

temporal_totals = df_raw.groupBy('CM_snapshot_date').count() \
    .orderBy('CM_snapshot_date')

display(temporal_totals)

# Calculate month-over-month changes
print("\n2. Month-over-Month Changes:")
print("-" * 70)

window_spec = Window.orderBy('CM_snapshot_date')

temporal_changes = temporal_totals \
    .withColumn('prev_count', lag('count').over(window_spec)) \
    .withColumn('change', col('count') - col('prev_count')) \
    .withColumn('pct_change', spark_round(((col('count') - col('prev_count')) / col('prev_count')) * 100, 2))

# Show all changes
display(temporal_changes)

# Identify largest changes
print("\n3. Largest Month-over-Month Changes:")
print("-" * 70)

print("\nTop 5 Increases:")
increases = temporal_changes.filter(col('change') > 0) \
    .orderBy(col('change').desc()).limit(5)
display(increases)

print("\nTop 5 Decreases:")
decreases = temporal_changes.filter(col('change') < 0) \
    .orderBy('change').limit(5)
display(decreases)

# Overall trend
print("\n4. Overall Temporal Trend:")
print("-" * 70)

first_snapshot = temporal_totals.orderBy('CM_snapshot_date').first()
last_snapshot = temporal_totals.orderBy(col('CM_snapshot_date').desc()).first()

total_change = last_snapshot['count'] - first_snapshot['count']
pct_change = (total_change / first_snapshot['count']) * 100

print(f"First snapshot ({first_snapshot['CM_snapshot_date']}): {first_snapshot['count']:,} records")
print(f"Last snapshot ({last_snapshot['CM_snapshot_date']}): {last_snapshot['count']:,} records")
print(f"Total change: {total_change:,} ({pct_change:+.2f}%)")

if total_change > 0:
    print("✓ Overall growth trend observed")
elif total_change < 0:
    print("⚠️ Overall decline trend observed")
else:
    print("✓ Stable membership over observation period")

TEMPORAL CONSISTENCY ANALYSIS

1. Total Records by Snapshot Date:
----------------------------------------------------------------------


CM_snapshot_date,count
2021-01-01,284197
2021-02-01,286668
2021-03-01,288309
2021-04-01,290440
2021-05-01,290890
2021-06-01,291313
2021-07-01,291633
2021-08-01,291910
2021-09-01,291734
2021-10-01,292540



2. Month-over-Month Changes:
----------------------------------------------------------------------


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CM_snapshot_date,count,prev_count,change,pct_change
2021-01-01,284197,null,null,null
2021-02-01,286668,284197,2471,0.87
2021-03-01,288309,286668,1641,0.57
2021-04-01,290440,288309,2131,0.74
2021-05-01,290890,290440,450,0.15
2021-06-01,291313,290890,423,0.15
2021-07-01,291633,291313,320,0.11
2021-08-01,291910,291633,277,0.09
2021-09-01,291734,291910,-176,-0.06
2021-10-01,292540,291734,806,0.28



3. Largest Month-over-Month Changes:
----------------------------------------------------------------------

Top 5 Increases:


CM_snapshot_date,count,prev_count,change,pct_change
2023-02-01,306759,301069,5690,1.89
2024-02-01,312305,309482,2823,0.91
2025-04-01,324329,321569,2760,0.86
2024-03-01,314880,312305,2575,0.82
2021-02-01,286668,284197,2471,0.87



Top 5 Decreases:


CM_snapshot_date,count,prev_count,change,pct_change
2024-01-01,309482,311394,-1912,-0.61
2025-01-01,318764,320231,-1467,-0.46
2025-12-01,324528,325825,-1297,-0.4
2023-09-01,312383,313676,-1293,-0.41
2022-01-01,290027,291272,-1245,-0.43



4. Overall Temporal Trend:
----------------------------------------------------------------------
First snapshot (2021-01-01): 284,197 records
Last snapshot (2025-12-01): 324,528 records
Total change: 40,331 (+14.19%)
✓ Overall growth trend observed


#### 2.7.1 Categorical Distribution Stability

In [0]:
# ═══════════════════════════════════════════════════════════════
# Categorical Distribution Over Time
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.window import Window

print("CATEGORICAL DISTRIBUTION STABILITY")
print("=" * 70)

print("\n1. MemCategory Distribution Over Time:")
print("-" * 70)

memcat_temporal = df_raw.groupBy('CM_snapshot_date', 'MemCategory').count() \
    .orderBy('CM_snapshot_date', 'MemCategory')

# Pivot to show categories side-by-side
memcat_pivot = memcat_temporal.groupBy('CM_snapshot_date').pivot('MemCategory').sum('count')
display(memcat_pivot)

print("\n" + "=" * 70)
print("2. MemSectorType Distribution Over Time:")
print("=" * 70)

sector_temporal = df_raw.groupBy('CM_snapshot_date', 'MemSectorType').count() \
    .orderBy('CM_snapshot_date', 'MemSectorType')

# Pivot to show sectors side-by-side
sector_pivot = sector_temporal.groupBy('CM_snapshot_date').pivot('MemSectorType').sum('count')
display(sector_pivot)

print("\n" + "=" * 70)
print("3. Region Distribution Stability Check:")
print("=" * 70)

# Check if regional proportions remain stable over time
region_temporal = df_raw.groupBy('CM_snapshot_date', 'Region').count()

# Calculate percentage of total for each region per snapshot
window_total = Window.partitionBy('CM_snapshot_date')

region_pct = region_temporal \
    .withColumn('total', spark_sum('count').over(window_total)) \
    .withColumn('percentage', spark_round((col('count') / col('total')) * 100, 2)) \
    .orderBy('CM_snapshot_date', 'Region')

# Show first and last snapshot for comparison
print("\nFirst snapshot (2021-01-01):")
first_regions = region_pct.filter(col('CM_snapshot_date') == '2021-01-01') \
    .select('Region', 'count', 'percentage').orderBy(col('percentage').desc())
display(first_regions)

print("\nLast snapshot (2025-12-01):")
last_regions = region_pct.filter(col('CM_snapshot_date') == '2025-12-01') \
    .select('Region', 'count', 'percentage').orderBy(col('percentage').desc())
display(last_regions)

CATEGORICAL DISTRIBUTION STABILITY

1. MemCategory Distribution Over Time:
----------------------------------------------------------------------


CM_snapshot_date,Nurse Support Worker,Nurse member,Nursing Support Worker,Student
2021-01-01,null,249188,22232,12777
2021-02-01,null,250477,22695,13496
2021-03-01,null,251330,23026,13953
2021-04-01,null,252628,23595,14217
2021-05-01,null,252832,23851,14207
2021-06-01,null,253037,24024,14252
2021-07-01,null,253173,24202,14258
2021-08-01,null,253504,24244,14162
2021-09-01,null,253488,24110,14136
2021-10-01,null,253925,24190,14425



2. MemSectorType Distribution Over Time:


CM_snapshot_date,null,Education,Independent,NHS,Other Public Sector,Other public sector
2021-01-01,24582,32151,74804,147718,997,3945
2021-02-01,24564,32499,75462,149183,1002,3958
2021-03-01,24532,32869,75859,150093,994,3962
2021-04-01,24575,32957,76299,151670,979,3960
2021-05-01,24474,32978,76476,152067,970,3925
2021-06-01,24420,32606,76584,152782,970,3951
2021-07-01,24418,32405,76794,153084,982,3950
2021-08-01,24288,32041,76983,153649,988,3961
2021-09-01,23923,31749,77125,153986,964,3987
2021-10-01,23942,31936,77161,154548,967,3986



3. Region Distribution Stability Check:

First snapshot (2021-01-01):


Region,count,percentage
South East,41302,14.53
London,37334,13.14
North West,27552,9.69
West Midlands,26917,9.47
South West,26624,9.37
Scotland,25137,8.84
Eastern,23965,8.43
East Midlands,19706,6.93
Yorkshire & The Humber,17388,6.12
Wales,16483,5.8



Last snapshot (2025-12-01):


Region,count,percentage
South East,46918,14.46
London,42180,13.0
West Midlands,31987,9.86
South West,30868,9.51
North West,30696,9.46
Scotland,28365,8.74
Eastern,26661,8.22
East Midlands,22376,6.89
Yorkshire & The Humber,20222,6.23
Wales,18729,5.77


**RESULT**

Temporal consistency analysis across 60 monthly snapshots (2021-01-01 to 2025-12-01) reveals stable growth with predictable seasonal patterns and consistent categorical distributions:

**Overall Temporal Trend:**
- Total records increased from 284,197 (January 2021) to 324,528 (December 2025)
- Net growth: 40,331 records (+14.19%) over five years
- Average monthly growth: approximately 672 records per month
- Steady, consistent expansion with minimal volatility
- Most monthly changes remain below 1% (±3,000 records)
- Largest increase: February 2023 (+5,690 records, +1.89%) - potential post-pandemic recovery, backlog processing, or policy change
- Largest decrease: January 2024 (-1,912 records, -0.61%)

**Seasonal Patterns:**
- January: Consistent membership decreases across multiple years (2022, 2024, 2025)
- February: Consistent membership increases, particularly strong in 2023-2024
- November-December: Minor declines suggesting year-end attrition
- Pattern indicates annual renewal cycle with end-of-year lapses and subsequent rejoins

**2.7.1 Categorical Distribution Stability:**

*MemCategory:*
- Nurse member category maintains 86-88% of total membership throughout observation period
- Support worker category naming change confirmed in May 2024: "Nursing Support Worker" (2021-2024) cleanly transitions to "Nurse Support Worker" (2024-2025) with no overlap
- Student category shows slight decline from 4.5% (early 2021) to 3.0% (late 2025)
- Proportions remain remarkably stable across all categories despite absolute growth

*MemSectorType:*
- NHS sector consistently represents 52-55% of total membership
- Independent sector maintains 26-27% throughout period
- Education sector shows gradual decline from 11.3% (January 2021) to 8.3% (December 2025), correlating with student category decline
- Other public sector remains stable at <2%
- Null values remain consistent at 8-9% across all snapshots

*Regional Distribution:*
- Regional proportions remain highly consistent across five-year period
- Top three regions maintain same rank order: South East (14.5%), London (13.0-13.1%), North West/West Midlands (9.5-9.9%)
- Smallest region (H Q Overseas) shows slight decline from 0.17% to 0.10%
- No significant geographic shifts or redistribution detected
- All 13 UK regions show proportional growth aligned with overall 14.19% expansion

**Key Findings:**
- Temporal data demonstrates exceptional stability and consistency across all categorical dimensions
- Growth is organic and steady with no sudden structural changes or anomalies
- Seasonal patterns are predictable and align with typical membership renewal cycles
- Category naming change in May 2024 was implemented cleanly without data disruption
- No evidence of data quality issues, missing snapshots, or temporal inconsistencies
- Regional and sectoral distributions maintain stable proportions despite absolute growth

**Status:** ✓ Pass

### 2.8 Outlier Identification

**CONTEXT**

Outlier identification systematically detects extreme or anomalous values in numerical variables that deviate significantly from expected patterns. These outliers may represent data entry errors, exceptional cases, or legitimate extreme values requiring investigation or special handling during analysis.

**PURPOSE**

To identify:
1. Statistical outliers in year variables (YoB, YoJ) using range-based thresholds
2. Extreme values in quantitative measures (q_members_t, q_leavers_t)
3. The magnitude and frequency of outlier occurrences
4. Whether outliers represent errors or legitimate edge cases

**STEP**

Apply threshold-based detection for year variables using domain knowledge (plausible birth/join years) and statistical methods for quantitative measures to identify, quantify, and characterize outlier records.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Outlier Identification
# ═══════════════════════════════════════════════════════════════

import builtins

print("OUTLIER IDENTIFICATION")
print("=" * 70)

# 1. YoB Outliers
print("\n1. YoB (Year of Birth) Outliers:")
print("-" * 70)

# Define plausible range: oldest possible member born 1900, youngest born 2010
yob_min_plausible = 1900
yob_max_plausible = 2010

yob_outliers_low = df_raw.filter(
    (col('YoB').isNotNull()) & 
    (col('YoB') < yob_min_plausible)
).count()

yob_outliers_high = df_raw.filter(
    (col('YoB').isNotNull()) & 
    (col('YoB') > yob_max_plausible)
).count()

total_yob_outliers = yob_outliers_low + yob_outliers_high

print(f"Plausible range: {yob_min_plausible} - {yob_max_plausible}")
print(f"Records with YoB < {yob_min_plausible}: {yob_outliers_low:,}")
print(f"Records with YoB > {yob_max_plausible}: {yob_outliers_high:,}")
print(f"Total YoB outliers: {total_yob_outliers:,} ({(total_yob_outliers/total_rows)*100:.4f}%)")

if total_yob_outliers > 0:
    print("\nDistinct outlier values:")
    outlier_vals = df_raw.filter(
        (col('YoB').isNotNull()) & 
        ((col('YoB') < yob_min_plausible) | (col('YoB') > yob_max_plausible))
    ).select('YoB').distinct().orderBy('YoB')
    display(outlier_vals)

# 2. YoJ Outliers
print("\n2. YoJ (Year of Join) Outliers:")
print("-" * 70)

# Define plausible range: the organisation founded 1916, future joins impossible
yoj_min_plausible = 1916
yoj_max_plausible = 2026  # Current year

yoj_outliers_low = df_raw.filter(
    (col('YoJ').isNotNull()) & 
    (col('YoJ') < yoj_min_plausible)
).count()

yoj_outliers_high = df_raw.filter(
    (col('YoJ').isNotNull()) & 
    (col('YoJ') > yoj_max_plausible)
).count()

total_yoj_outliers = yoj_outliers_low + yoj_outliers_high

print(f"Plausible range: {yoj_min_plausible} - {yoj_max_plausible}")
print(f"Records with YoJ < {yoj_min_plausible}: {yoj_outliers_low:,}")
print(f"Records with YoJ > {yoj_max_plausible}: {yoj_outliers_high:,}")
print(f"Total YoJ outliers: {total_yoj_outliers:,} ({(total_yoj_outliers/total_rows)*100:.4f}%)")

# 3. q_members_t Outliers (using statistical approach)
print("\n3. q_members_t Outliers:")
print("-" * 70)

# Get percentiles for statistical outlier detection
qmembers_stats = df_raw.select('q_members_t').summary('min', 'max', '25%', '50%', '75%')
display(qmembers_stats)

# Count records at extreme ends (top 1% and bottom 1%)
qmembers_p99 = df_raw.select('q_members_t').stat.approxQuantile('q_members_t', [0.99], 0.01)[0]
qmembers_p01 = df_raw.select('q_members_t').stat.approxQuantile('q_members_t', [0.01], 0.01)[0]

high_outliers = df_raw.filter(col('q_members_t') > qmembers_p99).count()
low_outliers = df_raw.filter(col('q_members_t') < qmembers_p01).count()

print(f"\n99th percentile: {qmembers_p99:.2f}")
print(f"1st percentile: {qmembers_p01:.2f}")
print(f"Records above 99th percentile: {high_outliers:,}")
print(f"Records below 1st percentile: {low_outliers:,}")

# 4. q_leavers_t Outliers (non-null only)
print("\n4. q_leavers_t Outliers:")
print("-" * 70)

qleavers_non_null = df_raw.filter(col('q_leavers_t').isNotNull())
qleavers_count = qleavers_non_null.count()

# Get percentiles
qleavers_stats = qleavers_non_null.select('q_leavers_t').summary('min', 'max', '25%', '50%', '75%')
display(qleavers_stats)

print(f"\nNote: Only {qleavers_count:,} non-null records ({(qleavers_count/total_rows)*100:.2f}%)")
print("With only 11 distinct values, all values represent the expected range")

OUTLIER IDENTIFICATION

1. YoB (Year of Birth) Outliers:
----------------------------------------------------------------------
Plausible range: 1900 - 2010
Records with YoB < 1900: 1
Records with YoB > 2010: 17
Total YoB outliers: 18 (0.0001%)

Distinct outlier values:


YoB
1895.0
2020.0
2021.0
2023.0
2024.0
2029.0
2042.0
2044.0
2966.0



2. YoJ (Year of Join) Outliers:
----------------------------------------------------------------------
Plausible range: 1916 - 2026
Records with YoJ < 1916: 0
Records with YoJ > 2026: 0
Total YoJ outliers: 0 (0.0000%)

3. q_members_t Outliers:
----------------------------------------------------------------------


summary,q_members_t
min,2.969121140142518
max,296.9121140142518
25%,2.969121140142518
50%,2.969121140142518
75%,5.938242280285036



99th percentile: 296.91
1st percentile: 2.97
Records above 99th percentile: 0
Records below 1st percentile: 0

4. q_leavers_t Outliers:
----------------------------------------------------------------------


summary,q_leavers_t
min,2.969121140142518
max,50.475059382422806
25%,2.969121140142518
50%,2.969121140142518
75%,2.969121140142518



Note: Only 202,475 non-null records (1.10%)
With only 11 distinct values, all values represent the expected range


**RESULT**

Outlier identification using threshold-based detection for year variables and statistical methods for quantitative measures reveals minimal outliers concentrated in year of birth data:

**YoB (Year of Birth) Outliers:**
- 18 outlier records identified (0.0001% of dataset)
- Low-end outliers: 1 record with YoB = 1895 (predates plausible range of 1900-2010)
- High-end outliers: 17 records with implausible future years (2020, 2021, 2023, 2024, 2029, 2042, 2044, 2966)
- Most extreme outlier: YoB = 2966 (940 years in the future)
- Consistent with findings from temporal logic validation (Section 2.6.2) identifying these as data entry errors likely stemming from legacy two-digit year system conversions

**YoJ (Year of Join) Outliers:**
- Zero outliers detected
- All 18,461,441 non-null records fall within plausible range (1916-2026)
- Clean data distribution with no extreme values
- Organisation's founding year (1916) appropriately used as minimum threshold

**q_members_t Outliers:**
- Statistical analysis reveals heavily concentrated distribution
- 75th percentile: 5.938 (multiplier ×2)
- 99th percentile: 296.91 (multiplier ×100, maximum value)
- Zero records exceed 99th percentile (expected given 99 distinct values from ×1 to ×100)
- Distribution confirms mathematical pattern: base value 2.969121 × multipliers (1-100)
- No statistical outliers - all values represent expected calculated range

**q_leavers_t Outliers:**
- Analysis limited to 202,475 non-null records (1.10% of dataset)
- Only 11 distinct values across entire range (2.97 to 50.48)
- Extreme concentration: 75th percentile equals median equals 25th percentile (2.97)
- Maximum value 50.48 (multiplier ×17) represents upper bound, not an outlier
- With only 11 discrete values, traditional outlier detection not applicable
- All values fall within expected calculated range using same base value as q_members_t

**Key Findings:**
- Outliers almost exclusively confined to YoB variable (18 records)
- YoJ data is clean with no outliers detected
- Quantitative measures (q_members_t, q_leavers_t) show no statistical outliers - all values conform to expected mathematical patterns
- Total outlier impact: 18 records (0.0001%) requiring correction
- Outlier patterns align with previously documented data quality issues (Sections 2.3, 2.6)
- No new or unexpected outliers discovered beyond those already identified in earlier validation steps

**Status:** ✓ Pass

### 2.9 Unexpected Value Detection

**CONTEXT**

Unexpected value detection identifies anomalous categorical values and patterns that violate business logic or data conventions. Unlike outlier detection which focuses on numerical extremes, this analysis targets unusual combinations, formatting inconsistencies, and values that should not logically coexist within the dataset structure.

**PURPOSE**

To identify:
1. Single-record anomalies (null, "Non Members", "Not Provided")
2. Formatting inconsistencies (casing variations in MemSectorType)
3. Invalid categorical combinations across related dimensions
4. Records that violate expected business rules or data collection standards

**STEP**

Systematically examine categorical variables for low-frequency anomalous values, formatting inconsistencies, and unexpected patterns previously identified during enumeration phases, quantifying their prevalence and impact.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Unexpected Value Detection
# ═══════════════════════════════════════════════════════════════

print("UNEXPECTED VALUE DETECTION")
print("=" * 70)

# 1. Single-record anomalies across categorical variables
print("\n1. Single-Record Anomalies:")
print("-" * 70)

anomalies_found = []

# Check Region
region_singles = df_raw.groupBy('Region').count().filter(col('count') == 1)
if region_singles.count() > 0:
    print("\nRegion single-record values:")
    display(region_singles)
    anomalies_found.append(('Region', region_singles.count()))

# Check Branch
branch_singles = df_raw.groupBy('Branch').count().filter(col('count') == 1)
if branch_singles.count() > 0:
    print("\nBranch single-record values:")
    display(branch_singles)
    anomalies_found.append(('Branch', branch_singles.count()))

# Check Int_nurse
int_nurse_singles = df_raw.groupBy('Int_nurse').count().filter(col('count') == 1)
if int_nurse_singles.count() > 0:
    print("\nInt_nurse single-record values:")
    display(int_nurse_singles)
    anomalies_found.append(('Int_nurse', int_nurse_singles.count()))

print(f"\nTotal columns with single-record anomalies: {len(anomalies_found)}")

# 2. MemSectorType casing inconsistency
print("\n2. MemSectorType Casing Inconsistency:")
print("-" * 70)

other_public_lower = df_raw.filter(col('MemSectorType') == 'Other public sector').count()
other_public_title = df_raw.filter(col('MemSectorType') == 'Other Public Sector').count()

print(f"'Other public sector' (lowercase): {other_public_lower:,} records")
print(f"'Other Public Sector' (title case): {other_public_title:,} records")
print(f"Combined total: {other_public_lower + other_public_title:,} records")
print(f"Impact: {(other_public_lower + other_public_title) / total_rows * 100:.2f}% of dataset")
print("⚠️ Inconsistent capitalization - should be standardized")

# 3. Null value summary across all columns
print("\n3. Null Value Summary Across Dataset:")
print("-" * 70)

null_summary = []
for col_name in df_raw.columns:
    null_count = df_raw.filter(col(col_name).isNull()).count()
    if null_count > 0:
        null_pct = (null_count / total_rows) * 100
        null_summary.append({
            'Column': col_name,
            'Null_Count': null_count,
            'Null_Percentage': round(null_pct, 4)
        })

null_summary_df = spark.createDataFrame(null_summary).orderBy(col('Null_Count').desc())
display(null_summary_df)

# 4. "Non Members" and "Not Provided" investigation
print("\n4. 'Non Members' and 'Not Provided' Investigation:")
print("-" * 70)

# Non Members in Region
non_members_region = df_raw.filter(col('Region') == 'Non Members')
if non_members_region.count() > 0:
    print("\n'Non Members' Region record details:")
    display(non_members_region.select('Region', 'Branch', 'MemCategory', 'MemSectorType', 
                                       'CM_snapshot_date', 'q_members_t').distinct())

# Non Members in Branch
non_members_branch = df_raw.filter(col('Branch') == 'NON MEMBERS')
if non_members_branch.count() > 0:
    print("\n'NON MEMBERS' Branch record details:")
    display(non_members_branch.select('Region', 'Branch', 'MemCategory', 'MemSectorType', 
                                       'CM_snapshot_date', 'q_members_t').distinct())

# Not Provided in Branch
not_provided_branch = df_raw.filter(col('Branch') == 'Not Provided')
if not_provided_branch.count() > 0:
    print("\n'Not Provided' Branch record details:")
    display(not_provided_branch.select('Region', 'Branch', 'MemCategory', 'MemSectorType', 
                                       'CM_snapshot_date', 'q_members_t').distinct())

print("\n✓ Unexpected value detection complete")

UNEXPECTED VALUE DETECTION

1. Single-Record Anomalies:
----------------------------------------------------------------------

Region single-record values:


Region,count
null,1
Non Members,1



Branch single-record values:


Branch,count
NON MEMBERS,1
Not Provided,1



Total columns with single-record anomalies: 2

2. MemSectorType Casing Inconsistency:
----------------------------------------------------------------------
'Other public sector' (lowercase): 267,728 records
'Other Public Sector' (title case): 39,493 records
Combined total: 307,221 records
Impact: 1.66% of dataset
⚠️ Inconsistent capitalization - should be standardized

3. Null Value Summary Across Dataset:
----------------------------------------------------------------------


...


'NON MEMBERS' Branch record details:


Region,Branch,MemCategory,MemSectorType,CM_snapshot_date,q_members_t
Non Members,NON MEMBERS,Nurse Support Worker,NHS,2025-01-01,2.969121140142518



'Not Provided' Branch record details:


Region,Branch,MemCategory,MemSectorType,CM_snapshot_date,q_members_t
null,Not Provided,Nurse Support Worker,NHS,2024-12-01,2.969121140142518



✓ Unexpected value detection complete


**RESULT**

Unexpected value detection reveals isolated anomalies concentrated in geographic classifications and a systematic formatting inconsistency in employment sector data:

**Single-Record Anomalies:**
- Two columns contain single-record anomalous values: Region (2 anomalies), Branch (2 anomalies)
- **Region anomalies**: "Non Members" (1 record), null (1 record)
- **Branch anomalies**: "NON MEMBERS" (1 record), "Not Provided" (1 record)
- Investigation reveals these represent two distinct problematic records:
 
 *Record 1 - "Non Members":*
 - Region: "Non Members", Branch: "NON MEMBERS"
 - Snapshot: 2025-01-01
 - Classification: Nurse Support Worker, NHS sector
 - Value: q_members_t = 2.97 (base multiplier)
 - Assessment: Likely data entry placeholder or test record
 
 *Record 2 - "Not Provided":*
 - Region: null, Branch: "Not Provided"
 - Snapshot: 2024-12-01
 - Classification: Nurse Support Worker, NHS sector
 - Value: q_members_t = 2.97 (base multiplier)
 - Assessment: Incomplete geographic assignment
 
- Both anomalies are recent (Dec 2024, Jan 2025), both categorized as Nurse Support Worker after May 2024 naming change, and both carry minimum quantitative values
- Combined impact: 2 records (0.00001% of dataset)

**MemSectorType Casing Inconsistency:**
- "Other public sector" (lowercase): 267,728 records
- "Other Public Sector" (title case): 39,493 records
- Combined total: 307,221 records (1.66% of dataset)
- Systematic formatting inconsistency requiring standardization during data cleaning
- Combined with 7.78% null values, MemSectorType has quality issues affecting 9.44% of records (1,743,916 total)

**Null Value Distribution:**
- q_leavers_t: 18,259,005 nulls (98.90%) - expected pattern for active members
- MemSectorType: 1,436,695 nulls (7.78%) - most problematic categorical variable
- YoB: 4,406 nulls (0.02%)
- YoJ: 39 nulls (0.0002%)
- Region: 1 null (part of "Not Provided" anomaly)
- Other columns: zero nulls
- Null distribution aligns with completeness analysis findings (Section 2.1)

**Cross-Column Validation:**
- "Non Members" appears identically in both Region and Branch columns for the same record, confirming data entry consistency within the anomaly
- Null Region value corresponds to "Not Provided" Branch value, suggesting intentional placeholder for missing geographic data
- Both anomalous records share identical characteristics (category, sector, value, timeframe), suggesting similar data collection circumstances

**Key Findings:**
- Unexpected values are minimal and isolated: 2 geographic anomalies (0.00001% of data)
- MemSectorType emerges as the most problematic categorical variable with 9.44% of records requiring attention (nulls + casing)
- Recent anomalies (Dec 2024 - Jan 2025) suggest ongoing data collection quality challenges
- No widespread systematic issues detected beyond documented casing inconsistency
- Anomalous records carry minimal analytical impact due to extremely low prevalence
- All unexpected values previously identified in categorical enumeration (Section 2.5) have been quantified and characterized

**Status:** ⚠️ Investigate

### 2.10 Data Quality Scorecard

**CONTEXT**

The data quality scorecard provides a comprehensive summary of all validation findings across completeness, consistency, accuracy, and integrity dimensions. This consolidated assessment quantifies the overall quality of the dataset and prioritizes issues requiring remediation before analysis.

**PURPOSE**

To synthesize:
1. Overall data quality metrics across all validation dimensions
2. Severity and prevalence of identified issues
3. Prioritized remediation requirements
4. Dataset readiness assessment for analytical use

**STEP**

Consolidate findings from all previous validation sections (2.1-2.9) into a structured scorecard with quantified metrics, issue categorization by severity, and actionable recommendations for data cleaning.

In [0]:
# ═══════════════════════════════════════════════════════════════
# Data Quality Scorecard
# ═══════════════════════════════════════════════════════════════

print("DATA QUALITY SCORECARD")
print("=" * 70)

# Summary metrics
print("\n" + "=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)

print(f"Total Records: {total_rows:,}")
print(f"Total Columns: {len(df_raw.columns)}")
print(f"Observation Period: 60 months (2021-01-01 to 2025-12-01)")
print(f"Data Structure: Aggregated membership snapshot panel")

# Completeness metrics
print("\n" + "=" * 70)
print("COMPLETENESS ASSESSMENT")
print("=" * 70)

completeness_summary = [
    {'Dimension': 'Columns with 0% missing', 'Count': 7, 'Percentage': 58.33},
    {'Dimension': 'Columns with <1% missing', 'Count': 3, 'Percentage': 25.00},
    {'Dimension': 'Columns with >1% missing', 'Count': 2, 'Percentage': 16.67},
    {'Dimension': 'Overall dataset completeness', 'Count': total_rows - 18259005, 'Percentage': 1.10}
]

completeness_df = spark.createDataFrame(completeness_summary)
display(completeness_df)

print("\nMost significant missingness:")
print("  • q_leavers_t: 98.90% (expected - active members)")
print("  • MemSectorType: 7.78% (requires investigation)")

# Data quality issues summary
print("\n" + "=" * 70)
print("IDENTIFIED DATA QUALITY ISSUES")
print("=" * 70)

quality_issues = [
    {'Issue': 'YoB future years (2020-2966)', 'Severity': 'High', 'Records': 17, 'Percentage': 0.0001},
    {'Issue': 'YoB pre-1900 values', 'Severity': 'Medium', 'Records': 1, 'Percentage': 0.0000},
    {'Issue': 'Birth/Join age violations (<16 years)', 'Severity': 'Medium', 'Records': 1183, 'Percentage': 0.0064},
    {'Issue': 'MemSectorType casing inconsistency', 'Severity': 'Medium', 'Records': 307221, 'Percentage': 1.66},
    {'Issue': 'MemSectorType null values', 'Severity': 'Low', 'Records': 1436695, 'Percentage': 7.78},
    {'Issue': 'Geographic anomalies (Non Members, Not Provided)', 'Severity': 'Low', 'Records': 2, 'Percentage': 0.0000},
    {'Issue': 'MemCategory naming change (documented)', 'Severity': 'Resolved', 'Records': 1729737, 'Percentage': 9.37}
]

quality_issues_df = spark.createDataFrame(quality_issues)
display(quality_issues_df)

print(f"\nTotal records requiring attention: {17 + 1 + 1183 + 307221 + 2:,} ({((17 + 1 + 1183 + 307221 + 2)/total_rows)*100:.2f}%)")
print("Note: MemSectorType nulls excluded as they may be legitimate missing data")

# Validation results summary
print("\n" + "=" * 70)
print("VALIDATION RESULTS BY SECTION")
print("=" * 70)

validation_results = [
    {'Section': '2.1 Completeness Analysis', 'Status': '⚠️ Investigate', 'Key Finding': 'q_leavers_t 98.9% null (expected), MemSectorType 7.8% null'},
    {'Section': '2.2 Cardinality Profiling', 'Status': '✓ Pass', 'Key Finding': 'Clear categorical structure with expected distinct counts'},
    {'Section': '2.3 Data Type Consistency', 'Status': '⚠️ Investigate', 'Key Finding': 'YoB/YoJ require integer casting, range violations detected'},
    {'Section': '2.4 Numerical Value Enumeration', 'Status': '⚠️ Investigate', 'Key Finding': 'Mathematical patterns confirmed, YoB outliers identified'},
    {'Section': '2.5 Categorical Value Enumeration', 'Status': '⚠️ Investigate', 'Key Finding': 'MemSectorType casing issue, 2 geographic anomalies'},
    {'Section': '2.6 Logical Consistency', 'Status': '✓ Pass', 'Key Finding': '1,183 temporal violations (0.0064%), otherwise clean'},
    {'Section': '2.7 Temporal Consistency', 'Status': '✓ Pass', 'Key Finding': 'Stable 14.19% growth, predictable seasonal patterns'},
    {'Section': '2.8 Outlier Identification', 'Status': '✓ Pass', 'Key Finding': '18 YoB outliers (0.0001%), no quantitative outliers'},
    {'Section': '2.9 Unexpected Value Detection', 'Status': '⚠️ Investigate', 'Key Finding': '2 geographic anomalies, MemSectorType issues confirmed'}
]

validation_df = spark.createDataFrame(validation_results)
display(validation_df)

# Overall quality score
print("\n" + "=" * 70)
print("OVERALL DATA QUALITY ASSESSMENT")
print("=" * 70)

total_issues = 17 + 1 + 1183 + 307221 + 2
issues_pct = (total_issues / total_rows) * 100

print(f"Records with quality issues: {total_issues:,} ({issues_pct:.2f}%)")
print(f"Clean records: {total_rows - total_issues:,} ({100 - issues_pct:.2f}%)")
print(f"\nOverall Quality Score: {100 - issues_pct:.2f}%")

if issues_pct < 2:
    quality_rating = "EXCELLENT"
elif issues_pct < 5:
    quality_rating = "GOOD"
elif issues_pct < 10:
    quality_rating = "ACCEPTABLE"
else:
    quality_rating = "REQUIRES IMPROVEMENT"

print(f"Quality Rating: {quality_rating}")
print("\n✓ Dataset is suitable for analysis with targeted cleaning")

DATA QUALITY SCORECARD

DATASET OVERVIEW
Total Records: 18,461,480
Total Columns: 12
Observation Period: 60 months (2021-01-01 to 2025-12-01)
Data Structure: Aggregated membership snapshot panel

COMPLETENESS ASSESSMENT


Count,Dimension,Percentage
7,Columns with 0% missing,58.33
3,Columns with <1% missing,25.0
2,Columns with >1% missing,16.67
202475,Overall dataset completeness,1.1



Most significant missingness:
  • q_leavers_t: 98.90% (expected - active members)
  • MemSectorType: 7.78% (requires investigation)

IDENTIFIED DATA QUALITY ISSUES


Issue,Percentage,Records,Severity
YoB future years (2020-2966),1.0E-4,17,High
YoB pre-1900 values,0.0,1,Medium
Birth/Join age violations (<16 years),0.0064,1183,Medium
MemSectorType casing inconsistency,1.66,307221,Medium
MemSectorType null values,7.78,1436695,Low
"Geographic anomalies (Non Members, Not Provided)",0.0,2,Low
MemCategory naming change (documented),9.37,1729737,Resolved



Total records requiring attention: 308,424 (1.67%)
Note: MemSectorType nulls excluded as they may be legitimate missing data

VALIDATION RESULTS BY SECTION


Key Finding,Section,Status
"q_leavers_t 98.9% null (expected), MemSectorType 7.8% null",2.1 Completeness Analysis,⚠️ Investigate
Clear categorical structure with expected distinct counts,2.2 Cardinality Profiling,✓ Pass
"YoB/YoJ require integer casting, range violations detected",2.3 Data Type Consistency,⚠️ Investigate
"Mathematical patterns confirmed, YoB outliers identified",2.4 Numerical Value Enumeration,⚠️ Investigate
"MemSectorType casing issue, 2 geographic anomalies",2.5 Categorical Value Enumeration,⚠️ Investigate
"1,183 temporal violations (0.0064%), otherwise clean",2.6 Logical Consistency,✓ Pass
"Stable 14.19% growth, predictable seasonal patterns",2.7 Temporal Consistency,✓ Pass
"18 YoB outliers (0.0001%), no quantitative outliers",2.8 Outlier Identification,✓ Pass
"2 geographic anomalies, MemSectorType issues confirmed",2.9 Unexpected Value Detection,⚠️ Investigate



OVERALL DATA QUALITY ASSESSMENT
Records with quality issues: 308,424 (1.67%)
Clean records: 18,153,056 (98.33%)

Overall Quality Score: 98.33%
Quality Rating: EXCELLENT

✓ Dataset is suitable for analysis with targeted cleaning


**RESULT**

The data quality scorecard synthesizes findings across nine validation dimensions, revealing an overall high-quality dataset with isolated, remediable issues:

**Dataset Overview:**
- 18,461,480 total records across 12 columns
- 60-month observation period (2021-01-01 to 2025-12-01)
- Aggregated membership snapshot panel structure
- Represents five years of continuous monthly data collection

**Completeness Assessment:**
- 7 columns (58.33%) have zero missing values
- 3 columns (25.00%) have minimal missingness (<1%)
- 2 columns (16.67%) have substantial missingness (>1%)
- q_leavers_t 98.90% null rate is expected and valid (represents active members)
- MemSectorType 7.78% null rate requires investigation as only problematic missingness
- Overall data completeness: 98.90% when excluding expected q_leavers_t nulls

**Issue Severity Classification:**

*High Severity (immediate correction required):*
- YoB future years (17 records, 0.0001%): Values 2020-2966 represent clear data entry errors

*Medium Severity (correction recommended):*
- Birth/Join age violations (1,183 records, 0.0064%): Likely legacy system conversion errors
- MemSectorType casing inconsistency (307,221 records, 1.66%): Systematic formatting issue
- YoB pre-1900 value (1 record): Single historical outlier

*Low Severity (minimal impact):*
- MemSectorType null values (1,436,695 records, 7.78%): May represent legitimate missing data
- Geographic anomalies (2 records, 0.0000%): Recent placeholder records

*Resolved Issues (documented, no action required):*
- MemCategory naming change (1,729,737 records, 9.37%): Clean systematic transition in May 2024

**Quality Metrics Summary:**
- Total records requiring correction: 308,424 (1.67%)
- Clean records: 18,153,056 (98.33%)
- **Overall Quality Score: 98.33%**
- **Quality Rating: EXCELLENT**

**Validation Performance:**
- 4 of 9 sections achieved ✓ Pass status (44.4%)
- 5 of 9 sections flagged for ⚠️ Investigation (55.6%)
- 0 sections received ❌ Critical failures (0%)
- Investigation flags primarily relate to documented, low-prevalence issues

**Prioritized Remediation Requirements:**

*Priority 1 (Critical):*
1. Correct 17 future birth year values (YoB 2020-2966)
2. Investigate and resolve 1 pre-1900 birth year value

*Priority 2 (Important):*
1. Standardize MemSectorType casing (convert all to lowercase or title case)
2. Cast YoB and YoJ from double to integer type
3. Investigate 1,183 birth/join age violations for pattern-based correction

*Priority 3 (Low):*
1. Investigate MemSectorType 7.78% null pattern
2. Resolve 2 geographic anomalies (Non Members, Not Provided)
3. Document MemCategory naming change for future reference

**Dataset Readiness Assessment:**
- **Status: READY for analysis with targeted cleaning**
- High overall quality score (98.33%) indicates dataset is fundamentally sound
- Identified issues are isolated, well-documented, and remediable
- No structural integrity problems detected
- Temporal consistency and hierarchical relationships validated
- Mathematical patterns in quantitative measures confirmed
- 98.33% of data requires no correction and is immediately usable

**Key Findings:**
- Dataset demonstrates exceptional quality with 98.33% of records clean and analysis-ready
- Issues are concentrated in specific, identifiable patterns (year conversions, casing, nulls)
- No widespread systematic data collection failures detected
- Temporal stability confirms consistent data quality over five-year period
- Hierarchical relationships (Region→Branch, MemCategory→CatName) maintain perfect integrity
- Quantitative measures follow expected mathematical patterns with no anomalies
- Dataset suitable for immediate analytical use while targeted cleaning addresses the 1.67% requiring correction

**Status:** ✓ Pass

## 3.0 Summary and Conclusions

This notebook established the foundational data environment and conducted comprehensive data quality assessment of the Organisation's membership churn dataset. Key accomplishments and findings are summarized below.

### 3.1 Data Ingestion Summary

Successfully ingested 2.06 GB CSV file containing 18,461,480 records across 12 columns, representing 60 monthly snapshots from January 2021 to December 2025. The dataset was validated for structural integrity, persisted in Delta format for efficient downstream processing, and confirmed to match expected dimensions. All data is now accessible in the Raw Data Table at `/Volumes/workspace/rcn_churn/raw_data/delta_raw/` for subsequent analysis phases.

### 3.2 Data Quality Assessment Summary

Systematic validation across nine dimensions revealed an overall quality score of **98.33%**, with 18,153,056 clean records and 308,424 records (1.67%) requiring targeted correction. The assessment identified:

- **18 critical issues**: YoB outliers with implausible future years requiring immediate correction
- **308,406 medium-priority issues**: Temporal logic violations (1,183), casing inconsistencies (307,221), and isolated anomalies (2)
- **1,436,695 low-priority issues**: MemSectorType null values requiring investigation

Key positive findings include perfect hierarchical integrity (Region→Branch, MemCategory→CatName), stable temporal patterns with 14.19% organic growth, and validated mathematical relationships in quantitative measures. The dataset is deemed **suitable for analysis** with targeted cleaning to address the identified 1.67% of problematic records.

### 3.3 Next Steps

**Notebook 02: Data Cleaning & Silver Layer Creation**
- Correct YoB outliers and temporal violations
- Standardize MemSectorType casing
- Cast YoB and YoJ to integer types
- Resolve geographic anomalies
- Create cleaned Silver Delta table

**Notebook 03: Exploratory Deep Dive**
- Correlation analysis
- Cohort analysis and retention patterns
- Churn rate calculations
- Member segmentation profiling
- Temporal trend analysis by segment

This foundational work provides a validated, documented baseline for all subsequent analytical phases.

**Corrected Membership Calculation Validation:**

Calculate total membership by summing q_members_t values rather than counting rows, as clarified by the Organisation's domain experts.

In [0]:
# Check what total membership actually is
from pyspark.sql.functions import sum as spark_sum

# By snapshot date
membership_by_date = df_raw.groupBy('CM_snapshot_date') \
    .agg(spark_sum('q_members_t').alias('total_members')) \
    .orderBy('CM_snapshot_date')

print("Total Membership by Snapshot (using SUM of q_members_t):")
display(membership_by_date)

# Overall total
total_membership = df_raw.select(spark_sum('q_members_t')).collect()[0][0]
print(f"\nTotal membership across all snapshots: {total_membership:,.2f}")
print(f"Average per snapshot: {total_membership/60:,.2f}")

Total Membership by Snapshot (using SUM of q_members_t):


CM_snapshot_date,total_members
2021-01-01,1436113.4204333434
2021-02-01,1438833.135397812
2021-03-01,1442595.0118824607
2021-04-01,1450219.7149704471
2021-05-01,1451573.6342092007
2021-06-01,1453132.4228089727
2021-07-01,1454694.180528713
2021-08-01,1456576.6033315775
2021-09-01,1458274.9406226422
2021-10-01,1472672.2090324669



Total membership across all snapshots: 96,940,014.85
Average per snapshot: 1,615,666.91


### 3.4 Important Methodological Note

Post-validation consultation with the Organisation confirmed that `q_members_t` represents aggregated membership units and must be analyzed using **SUM operations**, not row counts. Each row in this dataset represents a cohort segment (defined by snapshot date, region, branch, category, sector, birth year, and join year), and the value in `q_members_t` reflects the weighted membership count for that specific segment.

**Corrected Membership Understanding:**
- Dataset structure: 18,461,480 rows representing granular cohort segments
- True membership calculation: SUM(q_members_t) by snapshot date
- January 2021 membership: ~1.44 million members
- December 2025 membership: ~1.77 million members
- Five-year growth: +340,000 members (+23.7%)

**Implications for Analysis:**
- All data quality validations remain valid (structural integrity, nulls, outliers, consistency)
- Temporal trend analysis correctly identified growth pattern, though magnitude was misinterpreted
- Categorical distributions and hierarchical relationships unaffected
- Subsequent notebooks will use SUM(q_members_t) for all membership-based calculations
- Row-level data quality issues (18 YoB outliers, 1,183 temporal violations, etc.) remain accurate

This clarification demonstrates the importance of domain expert consultation in refining analytical approach while maintaining rigorous data quality standards.